In [8]:
# ============================================================
# SIH26028 — 02 DELAY EVOLUTION
# ASHWIN — RAILWAY DISRUPTION INTELLIGENCE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_DIR)

# ------------------------------------------------------------
# 2. Load delay dataset
# ------------------------------------------------------------

delay_file = DATA_DIR / "train_routes_delays_Sep2024.csv"

if not delay_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{delay_file}"
    )

df = pd.read_csv(delay_file)

# ------------------------------------------------------------
# 3. Validate
# ------------------------------------------------------------

required_columns = [
    "train",
    "date",
    "station",
    "sch_arr",
    "act_arr",
    "arr_delay",
    "sch_dep",
    "act_dep",
    "dep_delay"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# ------------------------------------------------------------
# 4. Prepare service date
# ------------------------------------------------------------

df["service_date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------

print("\n========== DATA LOADED ==========")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nUnique trains:", df["train"].nunique())
print("Unique dates:", df["service_date"].nunique())
print("Unique stations:", df["station"].nunique())

print("\nMissing values:")
print(df[required_columns].isna().sum())

print("\n========== READY ==========")
print("Dataframe created as: df")

Project root: c:\Users\Dell\Desktop\railway-intelligence
Raw data folder: c:\Users\Dell\Desktop\railway-intelligence\data\raw

========== DATA LOADED ==========
Shape: (1282325, 10)

Columns:
['train', 'date', 'station', 'sch_arr', 'act_arr', 'arr_delay', 'sch_dep', 'act_dep', 'dep_delay', 'service_date']

Unique trains: 3892
Unique dates: 30
Unique stations: 4736

Missing values:
train        0
date         0
station      0
sch_arr      0
act_arr      0
arr_delay    0
sch_dep      0
act_dep      0
dep_delay    0
dtype: int64

========== READY ==========
Dataframe created as: df


In [9]:
# ============================================================
# CELL 2 — PREPARE JOURNEY ORDER
# ============================================================

# Load route information for station sequence
route_file = DATA_DIR / "train_routes_Sep2024.csv"

routes = pd.read_csv(route_file)

print("Route data loaded:", routes.shape)
print("Route columns:", routes.columns.tolist())

# ------------------------------------------------------------
# Keep only the route information needed for ordering
# ------------------------------------------------------------

route_sequence = routes[
    [
        "trainNumber",
        "stnSerialNumber",
        "station_code",
        "distance"
    ]
].copy()

# Make train IDs consistent
route_sequence["trainNumber"] = (
    route_sequence["trainNumber"]
    .astype(str)
    .str.strip()
)

df["train"] = (
    df["train"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# Remove duplicate route definitions if any
# ------------------------------------------------------------

route_sequence = route_sequence.drop_duplicates(
    subset=["trainNumber", "stnSerialNumber"]
)

# ------------------------------------------------------------
# Merge station sequence into delay observations
# ------------------------------------------------------------

journey = df.merge(
    route_sequence,
    left_on=["train", "station"],
    right_on=["trainNumber", "station_code"],
    how="left"
)

# ------------------------------------------------------------
# Check merge quality
# ------------------------------------------------------------

matched = journey["stnSerialNumber"].notna().sum()
unmatched = journey["stnSerialNumber"].isna().sum()

print("\n========== JOURNEY MAPPING ==========")
print("Total delay observations:", len(journey))
print("Matched to route:", matched)
print("Unmatched to route:", unmatched)

print(
    "Match rate:",
    round(matched / len(journey) * 100, 2),
    "%"
)

# ------------------------------------------------------------
# Sort into train journey order
# ------------------------------------------------------------

journey = journey.sort_values(
    ["train", "service_date", "stnSerialNumber"]
).reset_index(drop=True)

print("\nJourney dataframe created.")
print("Rows:", len(journey))

print("\nSample — Train 12303:")
display(
    journey[
        (journey["train"] == "12303") &
        (journey["service_date"] == pd.Timestamp("2024-09-02"))
    ][
        [
            "train",
            "service_date",
            "stnSerialNumber",
            "station",
            "distance",
            "arr_delay",
            "dep_delay"
        ]
    ].head(30)
)

Route data loaded: (85055, 8)
Route columns: ['stnSerialNumber', 'trainNumber', 'trainName', 'station_code', 'station_name', 'distance', 'arrivalTime', 'departureTime']

========== JOURNEY MAPPING ==========
Total delay observations: 1283395
Matched to route: 1283313
Unmatched to route: 82
Match rate: 99.99 %

Journey dataframe created.
Rows: 1283395

Sample — Train 12303:


,train,service_date,stnSerialNumber,station,distance,arr_delay,dep_delay
146096,12303,2024-09-02,1.0,HWH,0.0,0.0,0.0
146097,12303,2024-09-02,2.0,BWN,95.0,15.0,15.0
146098,12303,2024-09-02,3.0,DGR,157.0,3.0,3.0
146099,12303,2024-09-02,4.0,ASN,199.0,2.0,2.0
146100,12303,2024-09-02,5.0,CRJ,224.0,4.0,4.0
146101,12303,2024-09-02,6.0,JMT,239.0,2.0,2.0
146102,12303,2024-09-02,7.0,MDP,281.0,2.0,2.0
146103,12303,2024-09-02,8.0,JSME,310.0,0.0,0.0
146104,12303,2024-09-02,9.0,JAJ,354.0,0.0,0.0
146105,12303,2024-09-02,10.0,JMU,380.0,4.0,4.0


In [10]:
# ============================================================
# CELL 3 — DELAY EVOLUTION FEATURES
# ============================================================

# Work only with the reconstructed journey data
evolution = journey.copy()

# Ensure correct chronological order
evolution = evolution.sort_values(
    ["train", "service_date", "stnSerialNumber"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 1. Change in arrival delay from previous station
# ------------------------------------------------------------

evolution["arr_delay_change"] = (
    evolution
    .groupby(["train", "service_date"])["arr_delay"]
    .diff()
)

# ------------------------------------------------------------
# 2. Change in departure delay from previous station
# ------------------------------------------------------------

evolution["dep_delay_change"] = (
    evolution
    .groupby(["train", "service_date"])["dep_delay"]
    .diff()
)

# ------------------------------------------------------------
# 3. Classify delay behaviour
# ------------------------------------------------------------

evolution["delay_behavior"] = np.select(
    [
        evolution["arr_delay_change"] > 2,
        evolution["arr_delay_change"] < -2
    ],
    [
        "AMPLIFICATION",
        "RECOVERY"
    ],
    default="STABLE"
)

print("========== DELAY EVOLUTION ==========")
print("Rows:", len(evolution))

print("\nBehavior counts:")
print(evolution["delay_behavior"].value_counts())

# ------------------------------------------------------------
# 4. Inspect Train 12303
# ------------------------------------------------------------

sample = evolution[
    (evolution["train"] == "12303") &
    (evolution["service_date"] == pd.Timestamp("2024-09-02"))
].copy()

print("\n========== TRAIN 12303 ==========")

display(
    sample[
        [
            "stnSerialNumber",
            "station",
            "distance",
            "arr_delay",
            "arr_delay_change",
            "delay_behavior",
            "dep_delay",
            "dep_delay_change"
        ]
    ]
)

========== DELAY EVOLUTION ==========
Rows: 1283395

Behavior counts:
delay_behavior
STABLE           610721
AMPLIFICATION    385808
RECOVERY         286866
Name: count, dtype: int64

========== TRAIN 12303 ==========


,stnSerialNumber,station,distance,arr_delay,arr_delay_change,delay_behavior,dep_delay,dep_delay_change
146096,1.0,HWH,0.0,0.0,NaN,STABLE,0.0,NaN
146097,2.0,BWN,95.0,15.0,15.0,AMPLIFICATION,15.0,15.0
146098,3.0,DGR,157.0,3.0,-12.0,RECOVERY,3.0,-12.0
146099,4.0,ASN,199.0,2.0,-1.0,STABLE,2.0,-1.0
146100,5.0,CRJ,224.0,4.0,2.0,STABLE,4.0,2.0
146101,6.0,JMT,239.0,2.0,-2.0,STABLE,2.0,-2.0
146102,7.0,MDP,281.0,2.0,0.0,STABLE,2.0,0.0
146103,8.0,JSME,310.0,0.0,-2.0,STABLE,0.0,-2.0
146104,9.0,JAJ,354.0,0.0,0.0,STABLE,0.0,0.0
146105,10.0,JMU,380.0,4.0,4.0,AMPLIFICATION,4.0,4.0


In [7]:
# ============================================================
# CELL 4 — SECTION-LEVEL DELAY EVOLUTION
# ============================================================

section = evolution.copy()

# ------------------------------------------------------------
# 1. Previous station
# ------------------------------------------------------------

section["previous_station"] = (
    section
    .groupby(["train", "service_date"])["station"]
    .shift(1)
)

section["previous_distance"] = (
    section
    .groupby(["train", "service_date"])["distance"]
    .shift(1)
)

# ------------------------------------------------------------
# 2. Section distance
# ------------------------------------------------------------

section["section_distance"] = (
    section["distance"] - section["previous_distance"]
)

# ------------------------------------------------------------
# 3. Keep only valid sections
# ------------------------------------------------------------

section_data = section[
    section["previous_station"].notna() &
    section["section_distance"].notna() &
    section["arr_delay_change"].notna()
].copy()

# ------------------------------------------------------------
# 4. Create section identifier
# ------------------------------------------------------------

section_data["section"] = (
    section_data["previous_station"].astype(str)
    + " -> "
    + section_data["station"].astype(str)
)

# ------------------------------------------------------------
# 5. Aggregate section behaviour
# ------------------------------------------------------------

section_stats = (
    section_data
    .groupby("section")
    .agg(
        observations=("arr_delay_change", "size"),
        mean_delay_change=("arr_delay_change", "mean"),
        median_delay_change=("arr_delay_change", "median"),
        std_delay_change=("arr_delay_change", "std"),
        mean_section_distance=("section_distance", "mean")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 6. Amplification / recovery rates
# ------------------------------------------------------------

section_data["amplified"] = (
    section_data["arr_delay_change"] > 2
)

section_data["recovered"] = (
    section_data["arr_delay_change"] < -2
)

rates = (
    section_data
    .groupby("section")
    .agg(
        amplification_rate=("amplified", "mean"),
        recovery_rate=("recovered", "mean")
    )
    .reset_index()
)

section_stats = section_stats.merge(
    rates,
    on="section",
    how="left"
)

# ------------------------------------------------------------
# 7. Reliability filter
# ------------------------------------------------------------

# Do not rank sections with only a handful of observations.
reliable_sections = section_stats[
    section_stats["observations"] >= 30
].copy()

# ------------------------------------------------------------
# 8. Results
# ------------------------------------------------------------

print("========== SECTION ANALYSIS ==========")

print("Total section observations:",
      len(section_data))

print("Unique sections:",
      section_data["section"].nunique())

print("Sections with >=30 observations:",
      len(reliable_sections))

print("\n========== TOP AMPLIFICATION SECTIONS ==========")

display(
    reliable_sections
    .sort_values(
        "mean_delay_change",
        ascending=False
    )
    .head(15)
)

print("\n========== TOP RECOVERY SECTIONS ==========")

display(
    reliable_sections
    .sort_values(
        "mean_delay_change",
        ascending=True
    )
    .head(15)
)

========== SECTION ANALYSIS ==========
Total section observations: 1225828
Unique sections: 17318
Sections with >=30 observations: 10673

========== TOP AMPLIFICATION SECTIONS ==========


,section,observations,mean_delay_change,median_delay_change,std_delay_change,mean_section_distance,amplification_rate,recovery_rate
7697,KGB -> BSP,30,211.266667,191.5,176.160768,32.000000,0.900000,0.000000
11808,NRKR -> NGP,38,170.447368,31.0,311.109149,84.000000,0.631579,0.078947
12939,PRYJ -> MKP,87,145.735632,0.0,244.369833,100.137931,0.448276,0.195402
7699,KGB -> PND,60,138.300000,29.5,171.969282,69.000000,0.733333,0.050000
15165,SPN -> MB,60,126.716667,37.5,148.962297,162.000000,0.683333,0.216667
14971,SMZ -> AMH,60,126.200000,0.0,394.952467,26.000000,0.150000,0.216667
4512,DMRX -> MKB,53,120.603774,90.0,102.736657,5.000000,0.830189,0.000000
3086,BYT -> R,192,103.989583,0.0,307.352365,63.833333,0.369792,0.255208
17212,YADD -> BG,46,100.130435,86.5,91.802350,6.391304,0.695652,0.000000
13311,RDL -> BBK,59,98.779661,-13.0,349.870324,61.050847,0.186441,0.694915



========== TOP RECOVERY SECTIONS ==========


,section,observations,mean_delay_change,median_delay_change,std_delay_change,mean_section_distance,amplification_rate,recovery_rate
2612,BSP -> KGB,30,-254.133333,-233.0,144.782770,32.000000,0.000000,1.000000
9742,MB -> SPN,60,-170.550000,-16.5,210.921901,161.000000,0.400000,0.500000
16679,VAK -> TVP,87,-152.781609,-13.0,321.751111,39.000000,0.000000,0.735632
11414,NGP -> NRKR,36,-135.305556,-31.5,260.941024,84.000000,0.027778,0.805556
16151,TNA -> CSMT,49,-118.387755,-13.0,174.593792,33.000000,0.122449,0.612245
12639,PND -> KGB,60,-104.616667,-4.0,164.491419,69.000000,0.266667,0.533333
350,ALER -> YADD,46,-101.217391,-92.0,93.956471,16.608696,0.000000,0.673913
11510,NIR -> JBP,39,-98.820513,-48.0,247.682296,123.000000,0.076923,0.794872
13640,RMNP -> BN,30,-92.666667,-93.0,19.540115,42.000000,0.000000,1.000000
1049,BBK -> RDL,46,-91.108696,-2.0,259.179494,61.000000,0.043478,0.434783


In [8]:
# ============================================================
# CELL 5 — INVESTIGATE EXTREME SECTION
# ============================================================

target_section = "KGB -> BSP"

extreme = section_data[
    section_data["section"] == target_section
].copy()

print("========== SECTION:", target_section, "==========")

print("Observations:", len(extreme))

display(
    extreme[
        [
            "train",
            "service_date",
            "previous_station",
            "station",
            "previous_distance",
            "distance",
            "section_distance",
            "arr_delay",
            "arr_delay_change",
            "dep_delay",
            "dep_delay_change"
        ]
    ]
    .sort_values(["service_date", "train"])
)

========== SECTION: KGB -> BSP ==========
Observations: 30


,train,service_date,previous_station,station,previous_distance,distance,section_distance,arr_delay,arr_delay_change,dep_delay,dep_delay_change
905520,18478,2024-09-01,KGB,BSP,1373.0,1405.0,32.0,346.0,346.0,346.0,346.0
905596,18478,2024-09-02,KGB,BSP,1373.0,1405.0,32.0,13.0,13.0,13.0,13.0
905672,18478,2024-09-03,KGB,BSP,1373.0,1405.0,32.0,0.0,0.0,0.0,0.0
905748,18478,2024-09-04,KGB,BSP,1373.0,1405.0,32.0,0.0,0.0,0.0,0.0
905824,18478,2024-09-05,KGB,BSP,1373.0,1405.0,32.0,290.0,290.0,290.0,290.0
905900,18478,2024-09-06,KGB,BSP,1373.0,1405.0,32.0,395.0,395.0,395.0,395.0
905976,18478,2024-09-07,KGB,BSP,1373.0,1405.0,32.0,159.0,159.0,159.0,159.0
906052,18478,2024-09-08,KGB,BSP,1373.0,1405.0,32.0,100.0,100.0,100.0,100.0
906128,18478,2024-09-09,KGB,BSP,1373.0,1405.0,32.0,254.0,254.0,254.0,254.0
906204,18478,2024-09-10,KGB,BSP,1373.0,1405.0,32.0,157.0,157.0,157.0,157.0


In [11]:
# ============================================================
# CELL 6 — TEMPORAL TRAIN INTERACTION FEASIBILITY
# ============================================================

# We need actual arrival timestamps for interaction analysis.

interaction = journey.copy()

# ------------------------------------------------------------
# Create actual arrival datetime
# ------------------------------------------------------------

interaction["actual_arr_dt"] = pd.to_datetime(
    interaction["service_date"].astype(str)
    + " "
    + interaction["act_arr"].astype(str),
    errors="coerce"
)

# ------------------------------------------------------------
# Fix midnight crossings within each train journey
# ------------------------------------------------------------

def fix_arrival_midnight(group):

    group = group.sort_values("stnSerialNumber").copy()

    values = group["actual_arr_dt"].tolist()

    for i in range(1, len(values)):

        if pd.notna(values[i]) and pd.notna(values[i - 1]):

            while values[i] < values[i - 1]:
                values[i] += pd.Timedelta(days=1)

    group["actual_arr_dt"] = values

    return group


interaction = (
    interaction
    .groupby(
        ["train", "service_date"],
        group_keys=False
    )
    .apply(fix_arrival_midnight)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Keep observations useful for interaction analysis
# ------------------------------------------------------------

interaction = interaction[
    interaction["actual_arr_dt"].notna() &
    interaction["station"].notna()
].copy()

print("========== INTERACTION DATA ==========")
print("Usable observations:", len(interaction))
print("Unique trains:", interaction["train"].nunique())
print("Unique stations:", interaction["station"].nunique())

print("\nTime range:")
print("Start:", interaction["actual_arr_dt"].min())
print("End:  ", interaction["actual_arr_dt"].max())

C:\Users\Dell\AppData\Local\Temp\ipykernel_5660\2971930971.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  interaction["actual_arr_dt"] = pd.to_datetime(
C:\Users\Dell\AppData\Local\Temp\ipykernel_5660\2971930971.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fix_arrival_midnight)


========== INTERACTION DATA ==========
Usable observations: 1283395
Unique trains: 3892
Unique stations: 4736

Time range:
Start: 2024-09-01 00:02:00
End:   2024-10-23 23:40:00


In [11]:
# ============================================================
# CELL 7 — SAME-STATION TEMPORAL INTERACTION FEASIBILITY
# ============================================================

# Sort observations by station and actual arrival time
station_time = interaction.sort_values(
    ["station", "actual_arr_dt"]
).copy()

# Previous train observed at the same station
station_time["prev_train"] = (
    station_time
    .groupby("station")["train"]
    .shift(1)
)

station_time["prev_arr_dt"] = (
    station_time
    .groupby("station")["actual_arr_dt"]
    .shift(1)
)

# Time gap from previous train at the same station
station_time["gap_minutes"] = (
    station_time["actual_arr_dt"]
    - station_time["prev_arr_dt"]
).dt.total_seconds() / 60

# Keep only interactions between different trains
interactions = station_time[
    (station_time["train"] != station_time["prev_train"]) &
    (station_time["gap_minutes"] >= 0)
].copy()

print("========== TEMPORAL INTERACTION ==========")

print("Potential train interactions:", len(interactions))

print(
    "Interactions within 5 min:",
    (interactions["gap_minutes"] <= 5).sum()
)

print(
    "Interactions within 10 min:",
    (interactions["gap_minutes"] <= 10).sum()
)

print(
    "Interactions within 15 min:",
    (interactions["gap_minutes"] <= 15).sum()
)

print(
    "Interactions within 30 min:",
    (interactions["gap_minutes"] <= 30).sum()
)

print("\nGap statistics:")
print(
    interactions["gap_minutes"]
    .describe()
)

print("\nUnique affected trains:")
print(interactions["train"].nunique())

print("\nUnique stations with interactions:")
print(interactions["station"].nunique())

========== TEMPORAL INTERACTION ==========
Potential train interactions: 1185750
Interactions within 5 min: 117944
Interactions within 10 min: 219927
Interactions within 15 min: 315224
Interactions within 30 min: 507838

Gap statistics:
count    1.185750e+06
mean     1.339703e+02
std      2.963736e+02
min      0.000000e+00
25%      1.500000e+01
50%      4.000000e+01
75%      1.240000e+02
max      2.653100e+04
Name: gap_minutes, dtype: float64

Unique affected trains:
3891

Unique stations with interactions:
4112


========== DELAY PROPAGATION SIGNAL ==========
                  interactions  mean_prev_delay  mean_curr_delay  median_curr_delay  curr_delay_gt_5  curr_delay_gt_15
prev_delay_class                                                                                                      
LOW                     146833         0.722494         0.722494                0.0              0.0               0.0
MODERATE                 51140        10.131032        10.131032               10.0              1.0               0.0
HIGH                     40328        22.073894        22.073894               22.0              1.0               1.0
SEVERE                   76923       138.717419       138.717419               76.0              1.0               1.0

Overall correlation:
                prev_arr_delay  curr_arr_delay
prev_arr_delay             1.0             1.0
curr_arr_delay             1.0             1.0


In [12]:
# ============================================================
# CELL 8 — VALID TRAIN-TO-TRAIN INTERACTION PAIRS
# LIGHT VERSION
# ============================================================

# We already created `interaction` in Cell 6.
# It contains:
# train, service_date, station, actual_arr_dt, arr_delay, dep_delay

# ------------------------------------------------------------
# 1. Sort observations by station and actual arrival time
# ------------------------------------------------------------

x = interaction[
    [
        "train",
        "service_date",
        "station",
        "actual_arr_dt",
        "arr_delay",
        "dep_delay"
    ]
].sort_values(
    ["station", "service_date", "actual_arr_dt"]
).copy()

# ------------------------------------------------------------
# 2. Look at the immediately preceding observation
#    at the same station/date
# ------------------------------------------------------------

x["source_train"] = (
    x.groupby(
        ["station", "service_date"]
    )["train"].shift(1)
)

x["source_arr_dt"] = (
    x.groupby(
        ["station", "service_date"]
    )["actual_arr_dt"].shift(1)
)

x["source_arr_delay"] = (
    x.groupby(
        ["station", "service_date"]
    )["arr_delay"].shift(1)
)

# ------------------------------------------------------------
# 3. Calculate time gap
# ------------------------------------------------------------

x["gap_minutes"] = (
    x["actual_arr_dt"] - x["source_arr_dt"]
).dt.total_seconds() / 60

# ------------------------------------------------------------
# 4. Keep different trains within 30 minutes
# ------------------------------------------------------------

pairs = x[
    (x["source_train"].notna()) &
    (x["source_train"] != x["train"]) &
    (x["gap_minutes"] > 0) &
    (x["gap_minutes"] <= 30)
].copy()

# Rename current train as target train
pairs = pairs.rename(
    columns={
        "train": "target_train",
        "actual_arr_dt": "target_arr_dt",
        "arr_delay": "target_arr_delay",
        "dep_delay": "target_dep_delay"
    }
)

# ------------------------------------------------------------
# 5. Classify source delay
# ------------------------------------------------------------

pairs["source_delay_class"] = pd.cut(
    pairs["source_arr_delay"],
    bins=[-float("inf"), 5, 15, 60, float("inf")],
    labels=["LOW", "MODERATE", "HIGH", "SEVERE"]
)

# ------------------------------------------------------------
# 6. Basic results
# ------------------------------------------------------------

print("========== VALID PROPAGATION PAIRS ==========")

print("Total pairs:", len(pairs))

print(
    "Unique source trains:",
    pairs["source_train"].nunique()
)

print(
    "Unique target trains:",
    pairs["target_train"].nunique()
)

print(
    "Unique stations:",
    pairs["station"].nunique()
)

print("\nGap statistics:")

print(
    pairs["gap_minutes"].describe()
)

# ------------------------------------------------------------
# 7. Show real examples
# ------------------------------------------------------------

print("\n========== SAMPLE TRAIN INTERACTIONS ==========")

display(
    pairs[
        [
            "service_date",
            "station",
            "source_train",
            "source_arr_dt",
            "source_arr_delay",
            "target_train",
            "target_arr_dt",
            "target_arr_delay",
            "gap_minutes",
            "source_delay_class"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 8. Basic delay comparison
# ------------------------------------------------------------

print("\n========== SOURCE DELAY VS TARGET DELAY ==========")

display(
    pairs.groupby("source_delay_class", observed=True)
    .agg(
        interactions=("target_train", "size"),
        mean_source_delay=("source_arr_delay", "mean"),
        mean_target_delay=("target_arr_delay", "mean"),
        median_target_delay=("target_arr_delay", "median")
    )
)

========== VALID PROPAGATION PAIRS ==========
Total pairs: 317387
Unique source trains: 3854
Unique target trains: 3860
Unique stations: 2116

Gap statistics:
count    317387.000000
mean         14.816665
std           8.255415
min           1.000000
25%           8.000000
50%          14.000000
75%          21.000000
max          30.000000
Name: gap_minutes, dtype: float64

========== SAMPLE TRAIN INTERACTIONS ==========


,service_date,station,source_train,source_arr_dt,source_arr_delay,target_train,target_arr_dt,target_arr_delay,gap_minutes,source_delay_class
89886,2024-09-01,AADR,14054,2024-09-01 21:20:00,2.0,12057,2024-09-01 21:46:00,0.0,26.0,LOW
90066,2024-09-11,AADR,14054,2024-09-11 21:21:00,3.0,12057,2024-09-11 21:48:00,2.0,27.0,LOW
90084,2024-09-12,AADR,14054,2024-09-12 21:20:00,2.0,12057,2024-09-12 21:50:00,4.0,30.0,LOW
90102,2024-09-13,AADR,14054,2024-09-13 21:20:00,2.0,12057,2024-09-13 21:47:00,0.0,27.0,LOW
90120,2024-09-14,AADR,14054,2024-09-14 21:20:00,2.0,12057,2024-09-14 21:46:00,0.0,26.0,LOW
90138,2024-09-15,AADR,14054,2024-09-15 21:21:00,3.0,12057,2024-09-15 21:50:00,4.0,29.0,LOW
90300,2024-09-24,AADR,14054,2024-09-24 21:19:00,0.0,12057,2024-09-24 21:47:00,0.0,28.0,LOW
90390,2024-09-29,AADR,14054,2024-09-29 21:22:00,4.0,12057,2024-09-29 21:52:00,6.0,30.0,LOW
90408,2024-09-30,AADR,14054,2024-09-30 21:20:00,2.0,12057,2024-09-30 21:47:00,0.0,27.0,LOW
598930,2024-09-01,AAL,12853,2024-09-01 23:56:00,13.0,15159,2024-09-02 00:25:00,37.0,29.0,MODERATE



========== SOURCE DELAY VS TARGET DELAY ==========


,interactions,mean_source_delay,mean_target_delay,median_target_delay
source_delay_class,,,,
LOW,141948,0.859801,27.145652,4.0
MODERATE,60670,10.137416,24.807763,8.0
HIGH,78390,30.283952,33.242633,13.0
SEVERE,36379,179.808516,52.512246,15.0


In [13]:
# ============================================================
# CELL 9 — TARGET TRAIN DOWNSTREAM DELAY CHANGE
# CORRECTED + LIGHT VERSION
# ============================================================

# We already have:
#   journey = complete train journey data
#   pairs   = source → target interaction pairs from Cell 8

# ------------------------------------------------------------
# 1. Build next-station information for each train journey
# ------------------------------------------------------------

target_journey = journey[
    [
        "train",
        "service_date",
        "stnSerialNumber",
        "station",
        "arr_delay",
        "dep_delay"
    ]
].sort_values(
    ["train", "service_date", "stnSerialNumber"]
).copy()

target_journey["next_station"] = (
    target_journey
    .groupby(["train", "service_date"])["station"]
    .shift(-1)
)

target_journey["next_arr_delay"] = (
    target_journey
    .groupby(["train", "service_date"])["arr_delay"]
    .shift(-1)
)

target_journey["next_dep_delay"] = (
    target_journey
    .groupby(["train", "service_date"])["dep_delay"]
    .shift(-1)
)

# ------------------------------------------------------------
# 2. Calculate delay change at the next station
# ------------------------------------------------------------

target_journey["downstream_arr_delay_change"] = (
    target_journey["next_arr_delay"]
    - target_journey["arr_delay"]
)

target_journey["downstream_dep_delay_change"] = (
    target_journey["next_dep_delay"]
    - target_journey["dep_delay"]
)

# ------------------------------------------------------------
# 3. Rename columns for merging
# ------------------------------------------------------------

target_features = target_journey[
    [
        "train",
        "service_date",
        "station",
        "next_station",
        "arr_delay",
        "dep_delay",
        "next_arr_delay",
        "next_dep_delay",
        "downstream_arr_delay_change",
        "downstream_dep_delay_change"
    ]
].rename(
    columns={
        "train": "target_train",
        "station": "station",
        "arr_delay": "target_current_arr_delay",
        "dep_delay": "target_current_dep_delay"
    }
)

# ------------------------------------------------------------
# 4. Attach downstream outcome to interaction pairs
# ------------------------------------------------------------

propagation = pairs.merge(
    target_features,
    on=[
        "target_train",
        "service_date",
        "station"
    ],
    how="left"
)

# ------------------------------------------------------------
# 5. Keep observations having a next station
# ------------------------------------------------------------

propagation = propagation[
    propagation["downstream_arr_delay_change"].notna()
].copy()

# ------------------------------------------------------------
# 6. Define amplification / recovery
# ------------------------------------------------------------

propagation["target_amplified"] = (
    propagation["downstream_arr_delay_change"] > 5
)

propagation["target_recovered"] = (
    propagation["downstream_arr_delay_change"] < -5
)

# ------------------------------------------------------------
# 7. RESULTS
# ------------------------------------------------------------

print("========== DOWNSTREAM PROPAGATION TEST ==========")

print(
    "Valid propagation observations:",
    len(propagation)
)

print(
    "Target delay amplified (>5 min):",
    round(propagation["target_amplified"].mean(), 4)
)

print(
    "Target delay recovered (>5 min):",
    round(propagation["target_recovered"].mean(), 4)
)

print("\nDownstream delay-change statistics:")

print(
    propagation["downstream_arr_delay_change"].describe()
)

# ------------------------------------------------------------
# 8. SOURCE DELAY CLASS COMPARISON
# ------------------------------------------------------------

print("\n========== SOURCE DELAY → DOWNSTREAM EFFECT ==========")

prop_summary = (
    propagation
    .groupby("source_delay_class", observed=True)
    .agg(
        observations=("target_train", "size"),

        mean_source_delay=(
            "source_arr_delay",
            "mean"
        ),

        mean_target_delay_change=(
            "downstream_arr_delay_change",
            "mean"
        ),

        median_target_delay_change=(
            "downstream_arr_delay_change",
            "median"
        ),

        amplification_rate=(
            "target_amplified",
            "mean"
        ),

        recovery_rate=(
            "target_recovered",
            "mean"
        )
    )
)

display(prop_summary)

# ------------------------------------------------------------
# 9. CORRELATION
# ------------------------------------------------------------

print("\n========== CORRELATION ==========")

print(
    propagation[
        [
            "source_arr_delay",
            "downstream_arr_delay_change"
        ]
    ].corr()
)

# ------------------------------------------------------------
# 10. REAL EXAMPLES
# ------------------------------------------------------------

print("\n========== SAMPLE PROPAGATION CASES ==========")

display(
    propagation[
        [
            "service_date",
            "station",
            "source_train",
            "source_arr_delay",
            "target_train",
            "target_current_arr_delay",
            "gap_minutes",
            "next_station",
            "next_arr_delay",
            "downstream_arr_delay_change",
            "source_delay_class"
        ]
    ].head(20)
)

========== DOWNSTREAM PROPAGATION TEST ==========
Valid propagation observations: 300628
Target delay amplified (>5 min): 0.2647
Target delay recovered (>5 min): 0.1675

Downstream delay-change statistics:
count    300628.000000
mean          1.554995
std          32.421903
min       -2308.000000
25%          -2.000000
50%           0.000000
75%           6.000000
max        2202.000000
Name: downstream_arr_delay_change, dtype: float64

========== SOURCE DELAY → DOWNSTREAM EFFECT ==========


,observations,mean_source_delay,mean_target_delay_change,median_target_delay_change,amplification_rate,recovery_rate
source_delay_class,,,,,,
LOW,131841,0.877049,2.144887,0.0,0.271479,0.144576
MODERATE,58259,10.144149,1.176934,0.0,0.241250,0.170514
HIGH,75881,30.263847,0.706343,0.0,0.255097,0.195675
SEVERE,34647,178.734407,1.804658,0.0,0.299247,0.188068



========== CORRELATION ==========
                             source_arr_delay  downstream_arr_delay_change
source_arr_delay                     1.000000                    -0.000632
downstream_arr_delay_change         -0.000632                     1.000000

========== SAMPLE PROPAGATION CASES ==========


,service_date,station,source_train,source_arr_delay,target_train,target_current_arr_delay,gap_minutes,next_station,next_arr_delay,downstream_arr_delay_change,source_delay_class
0,2024-09-01,AADR,14054,2.0,12057,0.0,26.0,DLPC,0.0,0.0,LOW
1,2024-09-11,AADR,14054,3.0,12057,2.0,27.0,DLPC,0.0,-2.0,LOW
2,2024-09-12,AADR,14054,2.0,12057,4.0,30.0,DLPC,0.0,-4.0,LOW
3,2024-09-13,AADR,14054,2.0,12057,0.0,27.0,DLPC,0.0,0.0,LOW
4,2024-09-14,AADR,14054,2.0,12057,0.0,26.0,DLPC,0.0,0.0,LOW
5,2024-09-15,AADR,14054,3.0,12057,4.0,29.0,DLPC,0.0,-4.0,LOW
6,2024-09-24,AADR,14054,0.0,12057,0.0,28.0,DLPC,0.0,0.0,LOW
7,2024-09-29,AADR,14054,4.0,12057,6.0,30.0,DLPC,0.0,-6.0,LOW
8,2024-09-30,AADR,14054,2.0,12057,0.0,27.0,DLPC,0.0,0.0,LOW
9,2024-09-01,AAL,12853,13.0,15159,37.0,29.0,APR,32.0,-5.0,MODERATE


In [14]:
# ============================================================
# CELL 10 — MULTI-STATION DELAY PROPAGATION
# ============================================================

# We already have:
#   journey = train journey data
#   pairs   = source → target interaction pairs
#
# We will measure the target train's delay change at:
#   +1 station
#   +2 stations
#   +3 stations
#
# This gives us a stronger propagation signal.

# ------------------------------------------------------------
# 1. Prepare target journey
# ------------------------------------------------------------

target = journey[
    [
        "train",
        "service_date",
        "stnSerialNumber",
        "station",
        "arr_delay"
    ]
].sort_values(
    ["train", "service_date", "stnSerialNumber"]
).copy()

# ------------------------------------------------------------
# 2. Delay at next 3 stations
# ------------------------------------------------------------

g = target.groupby(
    ["train", "service_date"]
)["arr_delay"]

target["delay_1"] = g.shift(-1)
target["delay_2"] = g.shift(-2)
target["delay_3"] = g.shift(-3)

# Current delay
target["current_delay"] = target["arr_delay"]

# ------------------------------------------------------------
# 3. Delay changes
# ------------------------------------------------------------

target["change_1"] = (
    target["delay_1"] - target["current_delay"]
)

target["change_2"] = (
    target["delay_2"] - target["current_delay"]
)

target["change_3"] = (
    target["delay_3"] - target["current_delay"]
)

# Maximum downstream increase
target["max_downstream_change"] = target[
    ["change_1", "change_2", "change_3"]
].max(axis=1)

# ------------------------------------------------------------
# 4. Propagation indicators
# ------------------------------------------------------------

target["propagated_5"] = (
    target["max_downstream_change"] > 5
)

target["propagated_15"] = (
    target["max_downstream_change"] > 15
)

# ------------------------------------------------------------
# 5. Rename for merging
# ------------------------------------------------------------

target = target.rename(
    columns={
        "train": "target_train",
        "station": "station"
    }
)

# ------------------------------------------------------------
# 6. Attach to interaction pairs
# ------------------------------------------------------------

multi_prop = pairs.merge(
    target[
        [
            "target_train",
            "service_date",
            "station",
            "current_delay",
            "delay_1",
            "delay_2",
            "delay_3",
            "change_1",
            "change_2",
            "change_3",
            "max_downstream_change",
            "propagated_5",
            "propagated_15"
        ]
    ],
    on=[
        "target_train",
        "service_date",
        "station"
    ],
    how="left"
)

# ------------------------------------------------------------
# 7. Remove incomplete downstream observations
# ------------------------------------------------------------

multi_prop = multi_prop[
    multi_prop["change_1"].notna()
].copy()

# ------------------------------------------------------------
# 8. Overall results
# ------------------------------------------------------------

print("========== MULTI-STATION PROPAGATION ==========")

print(
    "Valid interactions:",
    len(multi_prop)
)

print(
    "Propagation >5 min:",
    round(multi_prop["propagated_5"].mean(), 4)
)

print(
    "Propagation >15 min:",
    round(multi_prop["propagated_15"].mean(), 4)
)

print("\nMean delay change:")

print(
    multi_prop[
        ["change_1", "change_2", "change_3"]
    ].mean()
)

print("\nMedian delay change:")

print(
    multi_prop[
        ["change_1", "change_2", "change_3"]
    ].median()
)

# ------------------------------------------------------------
# 9. Compare source delay classes
# ------------------------------------------------------------

print("\n========== SOURCE DELAY CLASS ==========")

multi_summary = (
    multi_prop
    .groupby(
        "source_delay_class",
        observed=True
    )
    .agg(
        observations=("target_train", "size"),

        mean_source_delay=(
            "source_arr_delay",
            "mean"
        ),

        mean_change_1=(
            "change_1",
            "mean"
        ),

        mean_change_2=(
            "change_2",
            "mean"
        ),

        mean_change_3=(
            "change_3",
            "mean"
        ),

        propagation_5=(
            "propagated_5",
            "mean"
        ),

        propagation_15=(
            "propagated_15",
            "mean"
        )
    )
)

display(multi_summary)

# ------------------------------------------------------------
# 10. Correlations
# ------------------------------------------------------------

print("\n========== CORRELATIONS ==========")

print(
    multi_prop[
        [
            "source_arr_delay",
            "change_1",
            "change_2",
            "change_3",
            "max_downstream_change"
        ]
    ].corr()
)

========== MULTI-STATION PROPAGATION ==========
Valid interactions: 300628
Propagation >5 min: 0.4604
Propagation >15 min: 0.2416

Mean delay change:
change_1    1.554995
change_2    2.918286
change_3    3.797889
dtype: float64

Median delay change:
change_1    0.0
change_2    0.0
change_3    0.0
dtype: float64

========== SOURCE DELAY CLASS ==========


,observations,mean_source_delay,mean_change_1,mean_change_2,mean_change_3,propagation_5,propagation_15
source_delay_class,,,,,,,
LOW,131841,0.877049,2.144887,3.817347,4.906495,0.471955,0.241723
MODERATE,58259,10.144149,1.176934,2.336606,3.219553,0.440722,0.216739
HIGH,75881,30.263847,0.706343,1.595068,2.119099,0.443734,0.236014
SEVERE,34647,178.734407,1.804658,3.394130,4.243035,0.485987,0.295206



========== CORRELATIONS ==========
                       source_arr_delay  change_1  change_2  change_3  max_downstream_change
source_arr_delay               1.000000 -0.000632 -0.003719 -0.000631               0.022296
change_1                      -0.000632  1.000000  0.619337  0.506787               0.678189
change_2                      -0.003719  0.619337  1.000000  0.718757               0.750562
change_3                      -0.000631  0.506787  0.718757  1.000000               0.784228
max_downstream_change          0.022296  0.678189  0.750562  0.784228               1.000000


In [15]:
# ============================================================
# CELL 11 — PROPAGATION FEATURE DATASET
# ============================================================

# Create clean modelling dataset from multi_prop

features = multi_prop[
    [
        "service_date",
        "station",

        # Source train information
        "source_train",
        "source_arr_delay",
        "source_delay_class",

        # Target train state at interaction
        "target_train",
        "target_arr_delay",
        "target_dep_delay",
        "current_delay",

        # Temporal interaction
        "gap_minutes",

        # Downstream outcome information
        "change_1",
        "change_2",
        "change_3",
        "max_downstream_change",
        "propagated_5",
        "propagated_15"
    ]
].copy()


# ------------------------------------------------------------
# CREATE FINAL PREDICTION TARGETS
# ------------------------------------------------------------

features["target_propagation_5"] = (
    features["max_downstream_change"] > 5
).astype(int)

features["target_propagation_15"] = (
    features["max_downstream_change"] > 15
).astype(int)


# ------------------------------------------------------------
# BASIC CHECK
# ------------------------------------------------------------

print("========== PROPAGATION FEATURE DATASET ==========")

print("Rows:", len(features))
print("Columns:", len(features.columns))

print("\nMissing values:")
print(features.isna().sum())


# ------------------------------------------------------------
# TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\n========== TARGET DISTRIBUTION ==========")

print(
    "Propagation >5 min:",
    round(features["target_propagation_5"].mean(), 4)
)

print(
    "Propagation >15 min:",
    round(features["target_propagation_15"].mean(), 4)
)


# ------------------------------------------------------------
# SOURCE DELAY CLASSES
# ------------------------------------------------------------

print("\n========== SOURCE DELAY CLASSES ==========")

display(
    features["source_delay_class"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# PROPAGATION BY INTERACTION GAP
# ------------------------------------------------------------

features["gap_class"] = pd.cut(
    features["gap_minutes"],
    bins=[0, 5, 10, 15, 20, 30],
    labels=[
        "1-5 min",
        "6-10 min",
        "11-15 min",
        "16-20 min",
        "21-30 min"
    ],
    include_lowest=True
)

print("\n========== PROPAGATION BY TIME GAP ==========")

gap_summary = (
    features
    .groupby("gap_class", observed=True)
    .agg(
        observations=("target_train", "size"),
        propagation_5=("target_propagation_5", "mean"),
        propagation_15=("target_propagation_15", "mean"),
        mean_downstream_change=(
            "max_downstream_change",
            "mean"
        )
    )
)

display(gap_summary)


# ------------------------------------------------------------
# NUMERIC FEATURE SUMMARY
# ------------------------------------------------------------

print("\n========== NUMERIC FEATURES ==========")

display(
    features[
        [
            "source_arr_delay",
            "target_arr_delay",
            "target_dep_delay",
            "current_delay",
            "gap_minutes"
        ]
    ].describe()
)


# ------------------------------------------------------------
# FINAL MODELLING TABLE
# ------------------------------------------------------------

model_features = features[
    [
        "source_arr_delay",
        "source_delay_class",
        "target_arr_delay",
        "target_dep_delay",
        "current_delay",
        "gap_minutes",
        "station",
        "target_propagation_5",
        "target_propagation_15"
    ]
].copy()


print("\n========== MODEL DATASET READY ==========")

print("Rows:", len(model_features))

print("\nPredictors:")
print([
    c for c in model_features.columns
    if c not in [
        "target_propagation_5",
        "target_propagation_15"
    ]
])

print("\nTargets:")
print([
    "target_propagation_5",
    "target_propagation_15"
])

display(model_features.head(10))

========== PROPAGATION FEATURE DATASET ==========
Rows: 300628
Columns: 18

Missing values:
service_date                 0
station                      0
source_train                 0
source_arr_delay             0
source_delay_class           0
target_train                 0
target_arr_delay             0
target_dep_delay             0
current_delay                0
gap_minutes                  0
change_1                     0
change_2                 12917
change_3                 26289
max_downstream_change        0
propagated_5                 0
propagated_15                0
target_propagation_5         0
target_propagation_15        0
dtype: int64

========== TARGET DISTRIBUTION ==========
Propagation >5 min: 0.4604
Propagation >15 min: 0.2416

========== SOURCE DELAY CLASSES ==========


source_delay_class
LOW         131841
MODERATE     58259
HIGH         75881
SEVERE       34647
Name: count, dtype: int64


========== PROPAGATION BY TIME GAP ==========


,observations,propagation_5,propagation_15,mean_downstream_change
gap_class,,,,
1-5 min,48558,0.455991,0.245480,10.990547
6-10 min,56406,0.465376,0.249442,11.278126
11-15 min,59456,0.471811,0.249479,10.718767
16-20 min,51870,0.459938,0.239387,10.329034
21-30 min,84338,0.451837,0.229944,9.890133



========== NUMERIC FEATURES ==========


,source_arr_delay,target_arr_delay,target_dep_delay,current_delay,gap_minutes
count,300628.000000,300628.000000,300628.000000,300628.000000,300628.000000
mean,30.588239,31.744698,31.744698,31.744698,14.840298
std,78.879302,81.673422,81.673422,81.673422,8.247596
min,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,8.000000
50%,8.000000,8.000000,8.000000,8.000000,14.000000
75%,26.000000,26.000000,26.000000,26.000000,21.000000
max,2931.000000,2855.000000,2855.000000,2855.000000,30.000000



========== MODEL DATASET READY ==========
Rows: 300628

Predictors:
['source_arr_delay', 'source_delay_class', 'target_arr_delay', 'target_dep_delay', 'current_delay', 'gap_minutes', 'station']

Targets:
['target_propagation_5', 'target_propagation_15']


,source_arr_delay,source_delay_class,target_arr_delay,target_dep_delay,current_delay,gap_minutes,station,target_propagation_5,target_propagation_15
0,2.0,LOW,0.0,0.0,0.0,26.0,AADR,0,0
1,3.0,LOW,2.0,2.0,2.0,27.0,AADR,0,0
2,2.0,LOW,4.0,4.0,4.0,30.0,AADR,0,0
3,2.0,LOW,0.0,0.0,0.0,27.0,AADR,0,0
4,2.0,LOW,0.0,0.0,0.0,26.0,AADR,0,0
5,3.0,LOW,4.0,4.0,4.0,29.0,AADR,0,0
6,0.0,LOW,0.0,0.0,0.0,28.0,AADR,0,0
7,4.0,LOW,6.0,6.0,6.0,30.0,AADR,0,0
8,2.0,LOW,0.0,0.0,0.0,27.0,AADR,0,0
9,13.0,MODERATE,37.0,37.0,37.0,29.0,AAL,0,0


In [16]:
# ============================================================
# CELL 12 — TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

X = model_features[
    [
        "source_arr_delay",
        "source_delay_class",
        "target_arr_delay",
        "target_dep_delay",
        "current_delay",
        "gap_minutes",
        "station"
    ]
].copy()

# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

y = model_features["target_propagation_5"].copy()

# ------------------------------------------------------------
# Convert categorical columns
# ------------------------------------------------------------

X = pd.get_dummies(
    X,
    columns=["source_delay_class", "station"],
    drop_first=True
)

# ------------------------------------------------------------
# Train / test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("========== TRAIN / TEST SPLIT ==========")

print("Total observations:", len(X))

print("Training observations:", len(X_train))

print("Testing observations:", len(X_test))

print("Number of features:", X.shape[1])

print("\nTraining positive rate:",
      round(y_train.mean(), 4))

print("Testing positive rate:",
      round(y_test.mean(), 4))

print("\nReady for ML model.")

========== TRAIN / TEST SPLIT ==========
Total observations: 300628
Training observations: 240502
Testing observations: 60126
Number of features: 2113

Training positive rate: 0.4604
Testing positive rate: 0.4604

Ready for ML model.


In [22]:
# ============================================================
# CELL 13 — BASELINE PROPAGATION MODEL
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# Train baseline model
# ------------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Training Random Forest...")

model.fit(X_train, y_train)

print("Training complete.")


# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

print("\n========== BASELINE MODEL RESULTS ==========")

print(
    "Accuracy:",
    round(accuracy_score(y_test, y_pred), 4)
)

print(
    "Precision:",
    round(precision_score(y_test, y_pred), 4)
)

print(
    "Recall:",
    round(recall_score(y_test, y_pred), 4)
)

print(
    "F1 Score:",
    round(f1_score(y_test, y_pred), 4)
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, y_prob), 4)
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

print("\n========== CONFUSION MATRIX ==========")

print(
    confusion_matrix(y_test, y_pred)
)

Training Random Forest...
Training complete.

========== BASELINE MODEL RESULTS ==========
Accuracy: 0.5757
Precision: 0.5277
Recall: 0.7468
F1 Score: 0.6184
ROC-AUC: 0.6444

========== CONFUSION MATRIX ==========
[[13942 18502]
 [ 7008 20674]]


In [23]:
# ============================================================
# CELL 14 — FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})

importance = (
    importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("========== TOP 25 FEATURES ==========")

display(
    importance.head(25)
)

print("\n========== TOP NUMERIC FEATURES ==========")

numeric_features = [
    "source_arr_delay",
    "target_arr_delay",
    "target_dep_delay",
    "current_delay",
    "gap_minutes"
]

display(
    importance[
        importance["feature"].isin(numeric_features)
    ]
)

# ------------------------------------------------------------
# Total importance of station features
# ------------------------------------------------------------

station_importance = importance[
    importance["feature"].str.startswith("station_")
]["importance"].sum()

print(
    "\nTotal station feature importance:",
    round(station_importance, 4)
)

print(
    "Total non-station importance:",
    round(1 - station_importance, 4)
)

========== TOP 25 FEATURES ==========


,feature,importance
0,current_delay,0.156118
1,target_arr_delay,0.153367
2,target_dep_delay,0.121449
3,station_NDLS,0.044707
4,source_arr_delay,0.042645
5,gap_minutes,0.036866
6,station_CSMT,0.029975
7,station_MAS,0.019621
8,station_ASR,0.016961
9,station_BZA,0.015463



========== TOP NUMERIC FEATURES ==========


,feature,importance
0,current_delay,0.156118
1,target_arr_delay,0.153367
2,target_dep_delay,0.121449
4,source_arr_delay,0.042645
5,gap_minutes,0.036866



Total station feature importance: 0.4688
Total non-station importance: 0.5312


In [24]:
# ============================================================
# CELL 15 — FEATURE SANITY CHECK
# ============================================================

# current_delay, target_arr_delay and target_dep_delay
# appear to represent the same delay state in this dataset.
#
# Keep ONE representation to avoid giving the model
# duplicate information.

clean_model_features = features[
    [
        "source_arr_delay",
        "source_delay_class",

        # Target train's delay at interaction
        "target_arr_delay",

        # Temporal relationship
        "gap_minutes",

        # Location
        "station",

        # Prediction targets
        "target_propagation_5",
        "target_propagation_15"
    ]
].copy()


print("========== CLEAN FEATURE DATASET ==========")

print("Rows:", len(clean_model_features))

print("\nPredictors:")
print([
    c for c in clean_model_features.columns
    if c not in [
        "target_propagation_5",
        "target_propagation_15"
    ]
])

print("\nTargets:")
print([
    "target_propagation_5",
    "target_propagation_15"
])


# ------------------------------------------------------------
# Check whether the three old delay columns were identical
# ------------------------------------------------------------

print("\n========== DELAY CONSISTENCY CHECK ==========")

print(
    "current_delay == target_arr_delay:",
    (
        features["current_delay"]
        == features["target_arr_delay"]
    ).mean()
)

print(
    "target_arr_delay == target_dep_delay:",
    (
        features["target_arr_delay"]
        == features["target_dep_delay"]
    ).mean()
)

========== CLEAN FEATURE DATASET ==========
Rows: 300628

Predictors:
['source_arr_delay', 'source_delay_class', 'target_arr_delay', 'gap_minutes', 'station']

Targets:
['target_propagation_5', 'target_propagation_15']

========== DELAY CONSISTENCY CHECK ==========
current_delay == target_arr_delay: 1.0
target_arr_delay == target_dep_delay: 1.0


In [25]:
# ============================================================
# CELL 16 — CLEAN TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# Predictors
# ------------------------------------------------------------

X_clean = clean_model_features[
    [
        "source_arr_delay",
        "source_delay_class",
        "target_arr_delay",
        "gap_minutes",
        "station"
    ]
].copy()

# ------------------------------------------------------------
# Target: propagation >5 minutes
# ------------------------------------------------------------

y_clean = clean_model_features[
    "target_propagation_5"
].copy()

# ------------------------------------------------------------
# Encode categorical variables
# ------------------------------------------------------------

X_clean = pd.get_dummies(
    X_clean,
    columns=[
        "source_delay_class",
        "station"
    ],
    drop_first=True
)

# ------------------------------------------------------------
# Train / test split
# ------------------------------------------------------------

X_train_clean, X_test_clean, y_train_clean, y_test_clean = (
    train_test_split(
        X_clean,
        y_clean,
        test_size=0.20,
        random_state=42,
        stratify=y_clean
    )
)

print("========== CLEAN TRAIN / TEST SPLIT ==========")

print("Total observations:", len(X_clean))

print("Training observations:", len(X_train_clean))

print("Testing observations:", len(X_test_clean))

print("Number of features:", X_clean.shape[1])

print(
    "\nTraining propagation rate:",
    round(y_train_clean.mean(), 4)
)

print(
    "Testing propagation rate:",
    round(y_test_clean.mean(), 4)
)

print("\nClean dataset ready.")

========== CLEAN TRAIN / TEST SPLIT ==========
Total observations: 300628
Training observations: 240502
Testing observations: 60126
Number of features: 2111

Training propagation rate: 0.4604
Testing propagation rate: 0.4604

Clean dataset ready.


In [26]:
# ============================================================
# CELL 17 — CLEAN BASELINE RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

clean_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Training clean Random Forest...")

clean_model.fit(
    X_train_clean,
    y_train_clean
)

print("Training complete.")


# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_clean = clean_model.predict(X_test_clean)

y_prob_clean = clean_model.predict_proba(
    X_test_clean
)[:, 1]


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n========== CLEAN MODEL RESULTS ==========")

print(
    "Accuracy:",
    round(
        accuracy_score(y_test_clean, y_pred_clean),
        4
    )
)

print(
    "Precision:",
    round(
        precision_score(y_test_clean, y_pred_clean),
        4
    )
)

print(
    "Recall:",
    round(
        recall_score(y_test_clean, y_pred_clean),
        4
    )
)

print(
    "F1 Score:",
    round(
        f1_score(y_test_clean, y_pred_clean),
        4
    )
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(y_test_clean, y_prob_clean),
        4
    )
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

print("\n========== CONFUSION MATRIX ==========")

print(
    confusion_matrix(
        y_test_clean,
        y_pred_clean
    )
)


# ------------------------------------------------------------
# Direct comparison
# ------------------------------------------------------------

print("\n========== COMPARISON ==========")

print("Previous model ROC-AUC : 0.6444")

print(
    "Clean model ROC-AUC    :",
    round(
        roc_auc_score(y_test_clean, y_prob_clean),
        4
    )
)

Training clean Random Forest...
Training complete.

========== CLEAN MODEL RESULTS ==========
Accuracy: 0.5984
Precision: 0.5475
Recall: 0.7359
F1 Score: 0.6279
ROC-AUC: 0.6599

========== CONFUSION MATRIX ==========
[[15611 16833]
 [ 7311 20371]]

========== COMPARISON ==========
Previous model ROC-AUC : 0.6444
Clean model ROC-AUC    : 0.6599


In [30]:
# ============================================================
# CELL 12 — TEMPORAL VALIDATION
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Get dates WITHOUT changing the existing datasets
# ------------------------------------------------------------

dates_series = pd.to_datetime(
    features["service_date"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2. Find chronological cutoff
# ------------------------------------------------------------

unique_dates = sorted(
    dates_series.dropna().unique()
)

cutoff = int(len(unique_dates) * 0.80)

train_end = unique_dates[cutoff - 1]
test_start = unique_dates[cutoff]

# Boolean masks as NUMPY arrays
train_mask = (dates_series <= train_end).to_numpy()
test_mask = (dates_series >= test_start).to_numpy()

# ------------------------------------------------------------
# 3. Split X and y by POSITION
# ------------------------------------------------------------

X_train_temp = X.iloc[train_mask]
X_test_temp = X.iloc[test_mask]

y_train_temp = y.iloc[train_mask]
y_test_temp = y.iloc[test_mask]

print("========== TEMPORAL SPLIT ==========")

print("Training period:")
print(dates_series[train_mask].min(), "to", dates_series[train_mask].max())

print("\nTesting period:")
print(dates_series[test_mask].min(), "to", dates_series[test_mask].max())

print("\nTraining observations:", len(X_train_temp))
print("Testing observations:", len(X_test_temp))

print(
    "Training propagation rate:",
    round(y_train_temp.mean(), 4)
)

print(
    "Testing propagation rate:",
    round(y_test_temp.mean(), 4)
)

# ------------------------------------------------------------
# 4. Train
# ------------------------------------------------------------

print("\nTraining temporal Random Forest...")

temporal_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

temporal_rf.fit(
    X_train_temp,
    y_train_temp
)

print("Training complete.")

# ------------------------------------------------------------
# 5. Predict
# ------------------------------------------------------------

y_pred_temp = temporal_rf.predict(X_test_temp)

y_prob_temp = temporal_rf.predict_proba(
    X_test_temp
)[:, 1]

# ------------------------------------------------------------
# 6. Evaluate
# ------------------------------------------------------------

print("\n========== TEMPORAL MODEL RESULTS ==========")

print(
    "Accuracy:",
    round(accuracy_score(y_test_temp, y_pred_temp), 4)
)

print(
    "Precision:",
    round(precision_score(y_test_temp, y_pred_temp), 4)
)

print(
    "Recall:",
    round(recall_score(y_test_temp, y_pred_temp), 4)
)

print(
    "F1 Score:",
    round(f1_score(y_test_temp, y_pred_temp), 4)
)

temporal_auc = roc_auc_score(
    y_test_temp,
    y_prob_temp
)

print(
    "ROC-AUC:",
    round(temporal_auc, 4)
)

print("\n========== CONFUSION MATRIX ==========")

print(
    confusion_matrix(
        y_test_temp,
        y_pred_temp
    )
)

print("\n========== COMPARISON ==========")

print("Random-split ROC-AUC :", 0.6599)
print("Temporal ROC-AUC     :", round(temporal_auc, 4))

========== TEMPORAL SPLIT ==========
Training period:
2024-09-01 00:00:00 to 2024-09-24 00:00:00

Testing period:
2024-09-25 00:00:00 to 2024-09-30 00:00:00

Training observations: 239196
Testing observations: 61432
Training propagation rate: 0.4618
Testing propagation rate: 0.4548

Training temporal Random Forest...
Training complete.

========== TEMPORAL MODEL RESULTS ==========
Accuracy: 0.5393
Precision: 0.4955
Recall: 0.7309
F1 Score: 0.5907
ROC-AUC: 0.5894

========== CONFUSION MATRIX ==========
[[12708 20787]
 [ 7517 20420]]

========== COMPARISON ==========
Random-split ROC-AUC : 0.6599
Temporal ROC-AUC     : 0.5894


In [31]:
# ============================================================
# CELL 13 — TEMPORAL FEATURE IMPORTANCE
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Get feature importance from temporal Random Forest
# ------------------------------------------------------------

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": temporal_rf.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Top 25 features
# ------------------------------------------------------------

print("========== TEMPORAL MODEL — TOP 25 FEATURES ==========\n")

print(
    importance.head(25).to_string(index=False)
)

# ------------------------------------------------------------
# Numeric feature importance
# ------------------------------------------------------------

numeric_features = [
    "source_arr_delay",
    "target_arr_delay",
    "gap_minutes"
]

numeric_importance = importance[
    importance["feature"].isin(numeric_features)
]

print("\n========== NUMERIC FEATURE IMPORTANCE ==========\n")
print(numeric_importance.to_string(index=False))

# ------------------------------------------------------------
# Station importance
# ------------------------------------------------------------

station_importance = importance[
    importance["feature"].str.startswith("station_")
]["importance"].sum()

non_station_importance = (
    importance["importance"].sum()
    - station_importance
)

print("\n========== FEATURE GROUP IMPORTANCE ==========")

print(
    f"Station features:     {station_importance:.4f}"
)

print(
    f"Non-station features: {non_station_importance:.4f}"
)

========== TEMPORAL MODEL — TOP 25 FEATURES ==========

                    feature  importance
              current_delay    0.153138
           target_dep_delay    0.147238
           target_arr_delay    0.142392
                gap_minutes    0.048154
           source_arr_delay    0.046295
               station_NDLS    0.046029
               station_CSMT    0.040803
                station_ASR    0.021705
                station_MAS    0.014711
                station_BZA    0.012612
                station_NZM    0.011212
               station_ANVT    0.011026
               station_BINA    0.010043
                station_BRC    0.009846
    source_delay_class_HIGH    0.009482
                station_BBS    0.009272
               station_BEAS    0.008782
                station_LTT    0.008488
  source_delay_class_SEVERE    0.008366
                 station_BH    0.008093
               station_MMCT    0.007769
               station_BDTS    0.007653
               station_P

In [17]:
# ============================================================
# CELL 14 — CLEAN TEMPORAL FEATURE SET
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# Keep only non-redundant predictors
# ------------------------------------------------------------

keep_features = [
    "source_arr_delay",
    "target_arr_delay",
    "gap_minutes"
]

# Add one-hot encoded station + delay class features
keep_encoded = [
    col for col in X.columns
    if col.startswith("station_")
    or col.startswith("source_delay_class_")
]

X_clean_temporal = X[
    keep_features + keep_encoded
].copy()

print("========== CLEAN TEMPORAL DATASET ==========")
print("Rows:", len(X_clean_temporal))
print("Features:", X_clean_temporal.shape[1])

# ------------------------------------------------------------
# Recreate chronological split using propagation dataframe
# ------------------------------------------------------------

dates = pd.to_datetime(
    multi_prop["service_date"]
)

train_mask = dates <= pd.Timestamp("2024-09-24")
test_mask  = dates >= pd.Timestamp("2024-09-25")

X_train = X_clean_temporal.loc[train_mask.values]
X_test  = X_clean_temporal.loc[test_mask.values]

y_train = y.loc[train_mask.values]
y_test  = y.loc[test_mask.values]

print("\n========== TEMPORAL SPLIT ==========")
print("Training:", len(X_train))
print("Testing:", len(X_test))
print("Training propagation rate:", round(y_train.mean(), 4))
print("Testing propagation rate:", round(y_test.mean(), 4))

# ------------------------------------------------------------
# Train clean temporal Random Forest
# ------------------------------------------------------------

print("\nTraining clean temporal Random Forest...")

clean_temporal_rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

clean_temporal_rf.fit(X_train, y_train)

# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

pred = clean_temporal_rf.predict(X_test)
prob = clean_temporal_rf.predict_proba(X_test)[:, 1]

print("Training complete.")

print("\n========== CLEAN TEMPORAL MODEL RESULTS ==========")

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred), 4))
print("Recall:", round(recall_score(y_test, pred), 4))
print("F1 Score:", round(f1_score(y_test, pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))

print("\n========== CONFUSION MATRIX ==========")
print(confusion_matrix(y_test, pred))

print("\n========== COMPARISON ==========")
print("Previous temporal ROC-AUC:", 0.5894)
print(
    "Clean temporal ROC-AUC:",
    round(roc_auc_score(y_test, prob), 4)
)

========== CLEAN TEMPORAL DATASET ==========
Rows: 300628
Features: 2111

========== TEMPORAL SPLIT ==========
Training: 239196
Testing: 61432
Training propagation rate: 0.4586
Testing propagation rate: 0.4675

Training clean temporal Random Forest...
Training complete.

========== CLEAN TEMPORAL MODEL RESULTS ==========
Accuracy: 0.5844
Precision: 0.5403
Recall: 0.744
F1 Score: 0.626
ROC-AUC: 0.6503

========== CONFUSION MATRIX ==========
[[14534 18178]
 [ 7352 21368]]

========== COMPARISON ==========
Previous temporal ROC-AUC: 0.5894
Clean temporal ROC-AUC: 0.6503


In [33]:
# ============================================================
# CELL 15 — STRICT EARLY-WARNING MODEL
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Strict prediction features
# ------------------------------------------------------------

keep_features = [
    "source_arr_delay",
    "gap_minutes"
]

keep_encoded = [
    col for col in X.columns
    if col.startswith("station_")
    or col.startswith("source_delay_class_")
]

X_early = X[
    keep_features + keep_encoded
].copy()

print("========== EARLY-WARNING DATASET ==========")
print("Rows:", len(X_early))
print("Features:", X_early.shape[1])

# ------------------------------------------------------------
# 2. Chronological split
# ------------------------------------------------------------

dates = pd.to_datetime(multi_prop["service_date"])

train_mask = dates <= pd.Timestamp("2024-09-24")
test_mask  = dates >= pd.Timestamp("2024-09-25")

X_train = X_early.loc[train_mask.values]
X_test  = X_early.loc[test_mask.values]

y_train = y.loc[train_mask.values]
y_test  = y.loc[test_mask.values]

print("\n========== TEMPORAL SPLIT ==========")
print("Training:", len(X_train))
print("Testing:", len(X_test))
print("Training propagation rate:", round(y_train.mean(), 4))
print("Testing propagation rate:", round(y_test.mean(), 4))

# ------------------------------------------------------------
# 3. Train
# ------------------------------------------------------------

print("\nTraining early-warning Random Forest...")

early_rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

early_rf.fit(X_train, y_train)

# ------------------------------------------------------------
# 4. Evaluate
# ------------------------------------------------------------

pred = early_rf.predict(X_test)
prob = early_rf.predict_proba(X_test)[:, 1]

print("Training complete.")

print("\n========== EARLY-WARNING MODEL RESULTS ==========")

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred), 4))
print("Recall:", round(recall_score(y_test, pred), 4))
print("F1 Score:", round(f1_score(y_test, pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))

print("\n========== CONFUSION MATRIX ==========")
print(confusion_matrix(y_test, pred))

print("\n========== ROC-AUC COMPARISON ==========")
print("Original temporal model : 0.5894")
print("Clean temporal model    : 0.6503")
print(
    "Early-warning model     :",
    round(roc_auc_score(y_test, prob), 4)
)

========== EARLY-WARNING DATASET ==========
Rows: 300628
Features: 2110

========== TEMPORAL SPLIT ==========
Training: 239196
Testing: 61432
Training propagation rate: 0.4586
Testing propagation rate: 0.4675

Training early-warning Random Forest...
Training complete.

========== EARLY-WARNING MODEL RESULTS ==========
Accuracy: 0.5845
Precision: 0.5975
Recall: 0.3411
F1 Score: 0.4343
ROC-AUC: 0.6111

========== CONFUSION MATRIX ==========
[[26112  6600]
 [18924  9796]]

========== ROC-AUC COMPARISON ==========
Original temporal model : 0.5894
Clean temporal model    : 0.6503
Early-warning model     : 0.6111


In [36]:
# ============================================================
# CELL 16 — EARLY-WARNING MODEL
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("========== EARLY-WARNING DATASET ==========")

# ------------------------------------------------------------
# Use the clean propagation dataset
# ------------------------------------------------------------

ew_data = features.copy()

# Make sure target exists
if "target_propagation_5" not in ew_data.columns:
    raise KeyError(
        "target_propagation_5 missing from features. "
        "Available columns:\n" + str(ew_data.columns.tolist())
    )

# ------------------------------------------------------------
# Early-warning predictors
#
# IMPORTANT:
# Do NOT use current/target delay here.
# Those are too close to the outcome and create leakage.
# ------------------------------------------------------------

ew_predictors = [
    "source_arr_delay",
    "source_delay_class",
    "gap_minutes",
    "station"
]

ew_target = "target_propagation_5"

ew_data = ew_data[
    ew_predictors + [ew_target]
].copy()

ew_data = ew_data.dropna()

print("Rows:", len(ew_data))
print("Predictors:", ew_predictors)
print("Target:", ew_target)

# ------------------------------------------------------------
# Encode categorical variables
# ------------------------------------------------------------

X_ew = pd.get_dummies(
    ew_data[ew_predictors],
    columns=["source_delay_class", "station"],
    drop_first=False
)

y_ew = ew_data[ew_target].astype(int)

# ------------------------------------------------------------
# Recover chronological ordering
# ------------------------------------------------------------

# Use original row order for temporal split.
# 80% earliest observations = training
# 20% latest observations = testing
# ------------------------------------------------------------

split_idx = int(len(X_ew) * 0.80)

X_train_ew = X_ew.iloc[:split_idx].copy()
X_test_ew = X_ew.iloc[split_idx:].copy()

y_train_ew = y_ew.iloc[:split_idx].copy()
y_test_ew = y_ew.iloc[split_idx:].copy()

print("\n========== TEMPORAL SPLIT ==========")
print("Training:", len(X_train_ew))
print("Testing:", len(X_test_ew))

print(
    "Training propagation rate:",
    round(y_train_ew.mean(), 4)
)

print(
    "Testing propagation rate:",
    round(y_test_ew.mean(), 4)
)

# ------------------------------------------------------------
# Train early-warning Random Forest
# ------------------------------------------------------------

print("\nTraining early-warning Random Forest...")

early_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=14,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

early_model.fit(X_train_ew, y_train_ew)

print("Training complete.")

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

ew_pred = early_model.predict(X_test_ew)
ew_prob = early_model.predict_proba(X_test_ew)[:, 1]

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

accuracy = accuracy_score(y_test_ew, ew_pred)
precision = precision_score(y_test_ew, ew_pred, zero_division=0)
recall = recall_score(y_test_ew, ew_pred, zero_division=0)
f1 = f1_score(y_test_ew, ew_pred, zero_division=0)
auc = roc_auc_score(y_test_ew, ew_prob)

print("\n========== EARLY-WARNING MODEL RESULTS ==========")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("ROC-AUC:", round(auc, 4))

print("\n========== CONFUSION MATRIX ==========")
print(confusion_matrix(y_test_ew, ew_pred))

print("\n========== ROC-AUC COMPARISON ==========")
print("Original temporal model : 0.5894")
print("Clean temporal model    : 0.6503")
print("Early-warning model     :", round(auc, 4))

========== EARLY-WARNING DATASET ==========
Rows: 300628
Predictors: ['source_arr_delay', 'source_delay_class', 'gap_minutes', 'station']
Target: target_propagation_5

========== TEMPORAL SPLIT ==========
Training: 240502
Testing: 60126
Training propagation rate: 0.4586
Testing propagation rate: 0.4674

Training early-warning Random Forest...
Training complete.

========== EARLY-WARNING MODEL RESULTS ==========
Accuracy: 0.5859
Precision: 0.6007
Recall: 0.3403
F1 Score: 0.4345
ROC-AUC: 0.6131

========== CONFUSION MATRIX ==========
[[25666  6358]
 [18538  9564]]

========== ROC-AUC COMPARISON ==========
Original temporal model : 0.5894
Clean temporal model    : 0.6503
Early-warning model     : 0.6131


In [37]:
# ============================================================
# CELL 17 — EARLY-WARNING THRESHOLD ANALYSIS
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score

print("========== EARLY-WARNING THRESHOLD ANALYSIS ==========")

thresholds = [
    0.20, 0.25, 0.30, 0.35,
    0.40, 0.45, 0.50, 0.55,
    0.60, 0.65, 0.70
]

threshold_results = []

for threshold in thresholds:

    pred = (ew_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "warnings": int(pred.sum()),
        "warning_rate": pred.mean(),
        "precision": precision_score(
            y_test_ew, pred, zero_division=0
        ),
        "recall": recall_score(
            y_test_ew, pred, zero_division=0
        ),
        "f1": f1_score(
            y_test_ew, pred, zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

print(
    threshold_results.round(4).to_string(index=False)
)

# ------------------------------------------------------------
# Best F1 threshold
# ------------------------------------------------------------

best = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

print("\n========== BEST F1 THRESHOLD ==========")
print("Threshold:", best["threshold"])
print("Warnings:", best["warnings"])
print("Warning rate:", round(best["warning_rate"], 4))
print("Precision:", round(best["precision"], 4))
print("Recall:", round(best["recall"], 4))
print("F1:", round(best["f1"], 4))

========== EARLY-WARNING THRESHOLD ANALYSIS ==========
 threshold  warnings  warning_rate  precision  recall     f1
      0.20     60126        1.0000     0.4674  1.0000 0.6370
      0.25     60126        1.0000     0.4674  1.0000 0.6370
      0.30     60126        1.0000     0.4674  1.0000 0.6370
      0.35     60126        1.0000     0.4674  1.0000 0.6370
      0.40     60113        0.9998     0.4675  0.9999 0.6371
      0.45     58896        0.9795     0.4720  0.9893 0.6391
      0.50     15922        0.2648     0.6007  0.3403 0.4345
      0.55      1951        0.0324     0.8032  0.0558 0.1043
      0.60       274        0.0046     0.9307  0.0091 0.0180
      0.65         0        0.0000     0.0000  0.0000 0.0000
      0.70         0        0.0000     0.0000  0.0000 0.0000

========== BEST F1 THRESHOLD ==========
Threshold: 0.45
Warnings: 58896.0
Warning rate: 0.9795
Precision: 0.472
Recall: 0.9893
F1: 0.6391


In [38]:
# ============================================================
# CELL 18 — EARLY-WARNING PROBABILITY ANALYSIS
# ============================================================

print("========== PREDICTED PROBABILITY DISTRIBUTION ==========")

print(
    pd.Series(ew_prob).describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

# ------------------------------------------------------------
# Probability bands
# ------------------------------------------------------------

prob_bins = [
    0.00, 0.40, 0.45, 0.50,
    0.55, 0.60, 0.65, 1.01
]

prob_labels = [
    "<0.40",
    "0.40-0.45",
    "0.45-0.50",
    "0.50-0.55",
    "0.55-0.60",
    "0.60-0.65",
    ">=0.65"
]

prob_df = pd.DataFrame({
    "probability": ew_prob,
    "actual": y_test_ew.values
})

prob_df["risk_band"] = pd.cut(
    prob_df["probability"],
    bins=prob_bins,
    labels=prob_labels,
    right=False
)

band_summary = (
    prob_df
    .groupby("risk_band", observed=False)
    .agg(
        observations=("actual", "size"),
        actual_propagation_rate=("actual", "mean"),
        mean_predicted_probability=("probability", "mean")
    )
    .reset_index()
)

print("\n========== RISK BAND PERFORMANCE ==========")

print(
    band_summary.round(4).to_string(index=False)
)

========== PREDICTED PROBABILITY DISTRIBUTION ==========
count    60126.000000
mean         0.499587
std          0.020616
min          0.393969
1%           0.439258
5%           0.473900
10%          0.489855
25%          0.494143
50%          0.497470
75%          0.500980
90%          0.517535
95%          0.538319
99%          0.575760
max          0.631442
dtype: float64

========== RISK BAND PERFORMANCE ==========
risk_band  observations  actual_propagation_rate  mean_predicted_probability
    <0.40            13                   0.1538                      0.3973
0.40-0.45          1217                   0.2457                      0.4375
0.45-0.50         42974                   0.4244                      0.4936
0.50-0.55         13971                   0.5724                      0.5130
0.55-0.60          1677                   0.7823                      0.5670
0.60-0.65           274                   0.9307                      0.6152
   >=0.65             0             

In [39]:
# ============================================================
# CELL 19 — RISK SCORE × SOURCE DELAY CLASS
# ============================================================

print("========== RISK SCORE BY SOURCE DELAY CLASS ==========")

# Reconstruct test metadata from the same temporal split
ew_test_meta = ew_data.iloc[split_idx:].copy()

risk_eval = pd.DataFrame({
    "source_delay_class": ew_test_meta["source_delay_class"].values,
    "actual": y_test_ew.values,
    "score": ew_prob
})

# ------------------------------------------------------------
# Create operational risk bands
# ------------------------------------------------------------

risk_bins = [
    0.00, 0.45, 0.50,
    0.55, 0.60, 1.01
]

risk_labels = [
    "LOW",
    "MODERATE",
    "ELEVATED",
    "HIGH",
    "CRITICAL"
]

risk_eval["risk_band"] = pd.cut(
    risk_eval["score"],
    bins=risk_bins,
    labels=risk_labels,
    right=False
)

# ------------------------------------------------------------
# Propagation rate by delay class + risk band
# ------------------------------------------------------------

summary = (
    risk_eval
    .groupby(
        ["source_delay_class", "risk_band"],
        observed=False
    )
    .agg(
        observations=("actual", "size"),
        propagation_rate=("actual", "mean"),
        mean_score=("score", "mean")
    )
    .reset_index()
)

print(
    summary.round(4).to_string(index=False)
)

# ------------------------------------------------------------
# Overall risk-band summary
# ------------------------------------------------------------

print("\n========== OVERALL RISK BANDS ==========")

overall = (
    risk_eval
    .groupby("risk_band", observed=False)
    .agg(
        observations=("actual", "size"),
        propagation_rate=("actual", "mean"),
        mean_score=("score", "mean")
    )
    .reset_index()
)

print(
    overall.round(4).to_string(index=False)
)

========== RISK SCORE BY SOURCE DELAY CLASS ==========
source_delay_class risk_band  observations  propagation_rate  mean_score
               LOW       LOW           489            0.2454      0.4388
               LOW  MODERATE         19009            0.4216      0.4953
               LOW  ELEVATED          5204            0.6347      0.5176
               LOW      HIGH          1137            0.8004      0.5673
               LOW  CRITICAL           239            0.9331      0.6167
          MODERATE       LOW           411            0.2238      0.4346
          MODERATE  MODERATE          9766            0.4290      0.4910
          MODERATE  ELEVATED          1363            0.6449      0.5185
          MODERATE      HIGH           205            0.7463      0.5644
          MODERATE  CRITICAL            23            0.8696      0.6036
              HIGH       LOW           292            0.2705      0.4370
              HIGH  MODERATE         13604            0.4292      0.4

In [40]:
# ============================================================
# CELL 20 — STATION GENERALIZATION CHECK
# ============================================================

print("========== STATION GENERALIZATION ==========")

# Test how concentrated the predictions are by station

station_eval = pd.DataFrame({
    "station": ew_test_meta["station"].values,
    "actual": y_test_ew.values,
    "score": ew_prob
})

# ------------------------------------------------------------
# Station-level performance
# ------------------------------------------------------------

station_summary = (
    station_eval
    .groupby("station")
    .agg(
        observations=("actual", "size"),
        actual_propagation_rate=("actual", "mean"),
        mean_score=("score", "mean")
    )
    .reset_index()
)

# Only stations with enough observations
station_summary = station_summary[
    station_summary["observations"] >= 30
].copy()

print("Stations with >=30 test observations:",
      len(station_summary))

print("\nPropagation-rate range:")
print(
    station_summary["actual_propagation_rate"]
    .describe()
)

print("\n========== HIGHEST PROPAGATION STATIONS ==========")

print(
    station_summary
    .sort_values(
        "actual_propagation_rate",
        ascending=False
    )
    .head(15)
    .round(4)
    .to_string(index=False)
)

print("\n========== LOWEST PROPAGATION STATIONS ==========")

print(
    station_summary
    .sort_values(
        "actual_propagation_rate",
        ascending=True
    )
    .head(15)
    .round(4)
    .to_string(index=False)
)

# ------------------------------------------------------------
# Score correlation with actual station propagation
# ------------------------------------------------------------

if len(station_summary) > 5:

    station_corr = station_summary[
        [
            "actual_propagation_rate",
            "mean_score"
        ]
    ].corr().iloc[0, 1]

    print(
        "\nStation-level score correlation:",
        round(station_corr, 4)
    )

========== STATION GENERALIZATION ==========
Stations with >=30 test observations: 441

Propagation-rate range:
count    441.000000
mean       0.450622
std        0.164903
min        0.055556
25%        0.333333
50%        0.444444
75%        0.552239
max        1.000000
Name: actual_propagation_rate, dtype: float64

========== HIGHEST PROPAGATION STATIONS ==========
station  observations  actual_propagation_rate  mean_score
   BDTS            49                   1.0000      0.5713
    ASR            81                   0.9506      0.5975
   MMCT            39                   0.9487      0.5899
    LTT            56                   0.9464      0.5784
   ANVT            87                   0.9310      0.6098
   CSMT           208                   0.9038      0.6125
    SCM            33                   0.8485      0.4976
   INDB            73                   0.8219      0.5126
   BVRT            32                   0.8125      0.5042
   NDLS           516                   

In [42]:
# ============================================================
# FIX — GET CLEAN TEMPORAL MODEL PROBABILITIES
# ============================================================

temporal_clean_probs = clean_temporal_rf.predict_proba(
    X_test_clean
)[:, 1]

print("Probability predictions generated:", len(temporal_clean_probs))
print(
    "Range:",
    round(temporal_clean_probs.min(), 4),
    "to",
    round(temporal_clean_probs.max(), 4)
)

Probability predictions generated: 60126
Range: 0.4013 to 0.6097


In [43]:
# ============================================================
# CELL 18 — RISK SCORE CALIBRATION
# ============================================================

risk = pd.DataFrame({
    "actual": y_test_clean.values,
    "risk_score": temporal_clean_probs
})

# Create 10 approximately equal-sized risk groups
risk["risk_band"] = pd.qcut(
    risk["risk_score"],
    q=10,
    duplicates="drop"
)

calibration = (
    risk
    .groupby("risk_band", observed=True)
    .agg(
        observations=("actual", "size"),
        actual_propagation_rate=("actual", "mean"),
        mean_risk_score=("risk_score", "mean")
    )
    .reset_index()
)

print("========== CLEAN TEMPORAL RISK CALIBRATION ==========")
print(calibration.to_string(index=False))

# Calibration error
calibration["absolute_error"] = (
    calibration["actual_propagation_rate"]
    - calibration["mean_risk_score"]
).abs()

print("\n========== CALIBRATION QUALITY ==========")

print(
    "Mean absolute calibration error:",
    round(calibration["absolute_error"].mean(), 4)
)

print(
    "Maximum calibration error:",
    round(calibration["absolute_error"].max(), 4)
)

========== CLEAN TEMPORAL RISK CALIBRATION ==========
     risk_band  observations  actual_propagation_rate  mean_risk_score
   (0.4, 0.47]          6408                 0.208489         0.462974
 (0.47, 0.476]          5983                 0.308374         0.473415
(0.476, 0.482]          5647                 0.395608         0.476858
(0.482, 0.504]          6317                 0.377236         0.495490
(0.504, 0.505]          7118                 0.460663         0.504192
(0.505, 0.506]          4662                 0.500215         0.505314
(0.506, 0.507]          5969                 0.456190         0.506852
(0.507, 0.511]          5999                 0.563261         0.508902
 (0.511, 0.52]          6011                 0.594410         0.515424
  (0.52, 0.61]          6012                 0.764804         0.546309

========== CALIBRATION QUALITY ==========
Mean absolute calibration error: 0.107
Maximum calibration error: 0.2545


In [44]:
# ============================================================
# CELL 19 — PRACTICAL WARNING THRESHOLDS
# ============================================================

thresholds = [0.48, 0.50, 0.52, 0.54, 0.56, 0.58, 0.60]

results = []

for threshold in thresholds:

    predicted_warning = (
        temporal_clean_probs >= threshold
    )

    actual = y_test_clean.values

    warnings = predicted_warning.sum()

    if warnings > 0:
        precision = (
            actual[predicted_warning].mean()
        )
    else:
        precision = 0

    recall = (
        actual[predicted_warning].sum()
        / actual.sum()
    )

    warning_rate = (
        warnings / len(actual)
    )

    results.append({
        "threshold": threshold,
        "warnings": warnings,
        "warning_rate": warning_rate,
        "precision": precision,
        "recall": recall
    })

threshold_results = pd.DataFrame(results)

print("========== PRACTICAL WARNING THRESHOLDS ==========")

print(
    threshold_results.to_string(index=False)
)

print("\nInterpretation:")
print("Precision = fraction of warnings that actually propagated")
print("Recall    = fraction of all propagations successfully warned")
print("Warning rate = fraction of trains receiving a warning")

========== PRACTICAL WARNING THRESHOLDS ==========
 threshold  warnings  warning_rate  precision   recall
      0.48     42523      0.707231   0.528138 0.811285
      0.50     38309      0.637145   0.543267 0.751824
      0.52      6108      0.101587   0.764571 0.168702
      0.54      2937      0.048847   0.815458 0.086518
      0.56      1634      0.027176   0.852509 0.050322
      0.58       624      0.010378   0.931090 0.020988
      0.60        57      0.000948   0.982456 0.002023

Interpretation:
Precision = fraction of warnings that actually propagated
Recall    = fraction of all propagations successfully warned
Warning rate = fraction of trains receiving a warning


In [46]:
# CELL 19A — FIND TRAINED MODEL VARIABLES

from sklearn.ensemble import RandomForestClassifier

models_found = []

for name, obj in globals().items():
    if isinstance(obj, RandomForestClassifier):
        models_found.append(name)

print("Random Forest models currently in memory:")
for name in models_found:
    print(" -", name)

Random Forest models currently in memory:
 - model
 - clean_model
 - temporal_rf
 - clean_temporal_rf
 - early_rf
 - early_model


In [47]:
# ============================================================
# CELL 19 — FREEZE FINAL PROPAGATION MODEL
# ============================================================

import joblib
import os

# Final selected model
final_model = clean_temporal_rf

# Create model directory
os.makedirs("models", exist_ok=True)

# Save model
model_path = "models/propagation_rf_temporal.pkl"
joblib.dump(final_model, model_path)

# Save feature names
feature_names = list(X_train_clean.columns)

feature_path = "models/propagation_feature_names.pkl"
joblib.dump(feature_names, feature_path)

# Verify
print("========== FINAL PROPAGATION MODEL ==========")
print("Model:", type(final_model).__name__)
print("Features:", len(feature_names))
print("ROC-AUC: 0.6503")
print()
print("Model saved:", model_path)
print("Features saved:", feature_path)
print("STATUS: FROZEN")

========== FINAL PROPAGATION MODEL ==========
Model: RandomForestClassifier
Features: 2111
ROC-AUC: 0.6503

Model saved: models/propagation_rf_temporal.pkl
Features saved: models/propagation_feature_names.pkl
STATUS: FROZEN


In [48]:
# ============================================================
# CELL 20 — PROPAGATION RISK SCORER
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Get probabilities from the frozen model
# ------------------------------------------------------------

risk_scores = final_model.predict_proba(X_test_clean)[:, 1]

# ------------------------------------------------------------
# 2. Build risk dataframe
# ------------------------------------------------------------

risk_results = pd.DataFrame({
    "actual": y_test_clean.values,
    "risk_score": risk_scores
})

# ------------------------------------------------------------
# 3. Convert probability into operational risk bands
# ------------------------------------------------------------

def risk_band(score):
    if score < 0.40:
        return "LOW"
    elif score < 0.50:
        return "MODERATE"
    elif score < 0.55:
        return "ELEVATED"
    elif score < 0.60:
        return "HIGH"
    else:
        return "CRITICAL"

risk_results["risk_band"] = risk_results["risk_score"].apply(risk_band)

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

summary = (
    risk_results
    .groupby("risk_band", observed=False)
    .agg(
        observations=("actual", "size"),
        actual_propagation_rate=("actual", "mean"),
        mean_risk_score=("risk_score", "mean")
    )
)

# Keep bands in logical order
order = ["LOW", "MODERATE", "ELEVATED", "HIGH", "CRITICAL"]
summary = summary.reindex(order)

print("========== PROPAGATION RISK SCORER ==========")
print("Observations scored:", len(risk_results))
print()
print(summary.round(4))

print()
print("Risk scorer ready.")

========== PROPAGATION RISK SCORER ==========
Observations scored: 60126

           observations  actual_propagation_rate  mean_risk_score
risk_band                                                        
LOW                 NaN                      NaN              NaN
MODERATE        21817.0                   0.3149           0.4743
ELEVATED        36121.0                   0.5258           0.5102
HIGH             2131.0                   0.8282           0.5708
CRITICAL           57.0                   0.9825           0.6028

Risk scorer ready.


In [49]:
# ============================================================
# CELL 21 — STATION BOTTLENECK RANKING
# ============================================================

# Start from the temporal test observations
bottleneck = multi_prop.loc[
    X_test_clean.index,
    [
        "station",
        "source_train",
        "target_train",
        "source_arr_delay",
        "gap_minutes"
    ]
].copy()

# Attach model risk score
bottleneck["risk_score"] = risk_scores

# Actual outcome
bottleneck["propagated"] = y_test_clean.values

# ------------------------------------------------------------
# 1. Station-level aggregation
# ------------------------------------------------------------

station_risk = (
    bottleneck
    .groupby("station")
    .agg(
        observations=("risk_score", "size"),
        propagation_rate=("propagated", "mean"),
        mean_risk_score=("risk_score", "mean"),
        mean_source_delay=("source_arr_delay", "mean"),
        mean_gap=("gap_minutes", "mean")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 2. Remove very small samples
# ------------------------------------------------------------

station_risk = station_risk[
    station_risk["observations"] >= 30
].copy()

# ------------------------------------------------------------
# 3. Create a simple bottleneck score
# ------------------------------------------------------------

station_risk["bottleneck_score"] = (
    station_risk["mean_risk_score"]
    * station_risk["propagation_rate"]
)

# ------------------------------------------------------------
# 4. Rank stations
# ------------------------------------------------------------

station_risk = station_risk.sort_values(
    "bottleneck_score",
    ascending=False
)

print("========== TOP NETWORK BOTTLENECKS ==========")

print(
    station_risk[
        [
            "station",
            "observations",
            "propagation_rate",
            "mean_risk_score",
            "mean_source_delay",
            "mean_gap",
            "bottleneck_score"
        ]
    ]
    .head(20)
    .round(4)
    .to_string(index=False)
)

print()
print("Stations evaluated:", len(station_risk))

========== TOP NETWORK BOTTLENECKS ==========
station  observations  propagation_rate  mean_risk_score  mean_source_delay  mean_gap  bottleneck_score
    ASR            91            0.9670           0.5906             9.1648   20.7912            0.5711
   CSMT           199            0.9095           0.5902            10.1407   13.6834            0.5369
   ANVT            84            0.9048           0.5884            20.6071   15.5833            0.5324
   MMCT            52            0.9615           0.5324             1.6731   17.1923            0.5119
   BDTS            46            0.8913           0.5349             1.4783   17.2174            0.4767
    LTT            48            0.8542           0.5541            41.9167   17.0000            0.4733
    TVC            70            0.7714           0.5738             6.4714   14.7571            0.4426
    MAO            45            0.8000           0.5492            42.5778   16.7778            0.4394
    MYS           

In [50]:
# ============================================================
# CELL 22 — TOP RISKY TRAIN INTERACTIONS
# ============================================================

# Keep the original interaction information
interaction_risk = multi_prop.loc[
    X_test_clean.index,
    [
        "service_date",
        "station",
        "source_train",
        "target_train",
        "source_arr_delay",
        "target_arr_delay",
        "gap_minutes"
    ]
].copy()

# Add model outputs
interaction_risk["risk_score"] = risk_scores
interaction_risk["propagated"] = y_test_clean.values

# ------------------------------------------------------------
# 1. Focus on genuinely high-risk interactions
# ------------------------------------------------------------

high_risk = interaction_risk[
    interaction_risk["risk_score"] >= 0.55
].copy()

# ------------------------------------------------------------
# 2. Rank by risk score
# ------------------------------------------------------------

high_risk = high_risk.sort_values(
    "risk_score",
    ascending=False
)

print("========== HIGH-RISK TRAIN INTERACTIONS ==========")
print(
    high_risk[
        [
            "service_date",
            "station",
            "source_train",
            "target_train",
            "source_arr_delay",
            "target_arr_delay",
            "gap_minutes",
            "risk_score",
            "propagated"
        ]
    ]
    .head(25)
    .round(4)
    .to_string(index=False)
)

print()
print("High-risk interactions:", len(high_risk))
print(
    "Observed propagation among high-risk interactions:",
    round(high_risk["propagated"].mean(), 4)
)

========== HIGH-RISK TRAIN INTERACTIONS ==========
service_date station source_train target_train  source_arr_delay  target_arr_delay  gap_minutes  risk_score  propagated
  2024-09-17    CSMT        22159        20706              24.0               2.0          3.0      0.6097           1
  2024-09-29    CSMT        12072        12261              28.0               4.0          1.0      0.6083           1
  2024-09-11    CSMT        22105        12859              19.0               3.0          4.0      0.6062           1
  2024-09-21    NDLS        12034         2394               3.0             161.0          8.0      0.6053           1
  2024-09-11    CSMT        22106        12322              22.0              11.0          4.0      0.6052           1
  2024-09-05    NDLS        14728         2394               3.0             306.0         15.0      0.6049           1
  2024-09-22    NDLS        20805        20808             115.0               2.0         27.0      0.6049  

In [52]:
# ============================================================
# CELL 18 — OPERATIONAL PROPAGATION RISK SCORING
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Use the frozen CLEAN TEMPORAL model
# ------------------------------------------------------------

final_model = clean_temporal_rf

# ------------------------------------------------------------
# 2. Get the actual temporal test rows
# ------------------------------------------------------------

# IMPORTANT:
# Do NOT align using .loc[operational.index].
# The temporal split already created the correct test matrices.

X_test_final = X_test_clean.copy()
y_test_final = y_test_clean.copy()

# Reset indices so everything aligns positionally
X_test_final = X_test_final.reset_index(drop=True)
y_test_final = y_test_final.reset_index(drop=True)

# ------------------------------------------------------------
# 3. Generate propagation probabilities
# ------------------------------------------------------------

risk_scores = final_model.predict_proba(X_test_final)[:, 1]

# ------------------------------------------------------------
# 4. Create operational dataframe
# ------------------------------------------------------------

operational = pd.DataFrame({
    "actual_propagation": y_test_final.values,
    "risk_score": risk_scores
})

# ------------------------------------------------------------
# 5. Create operational risk bands
# ------------------------------------------------------------

def risk_band(score):

    if score < 0.47:
        return "LOW"

    elif score < 0.50:
        return "MODERATE"

    elif score < 0.55:
        return "ELEVATED"

    elif score < 0.60:
        return "HIGH"

    else:
        return "CRITICAL"


operational["risk_band"] = operational["risk_score"].apply(risk_band)

# ------------------------------------------------------------
# 6. Risk-band performance
# ------------------------------------------------------------

risk_summary = (
    operational
    .groupby("risk_band", observed=True)
    .agg(
        observations=("actual_propagation", "size"),
        actual_propagation_rate=("actual_propagation", "mean"),
        mean_risk_score=("risk_score", "mean")
    )
)

# Keep logical order
risk_order = [
    "LOW",
    "MODERATE",
    "ELEVATED",
    "HIGH",
    "CRITICAL"
]

risk_summary = risk_summary.reindex(risk_order)

print("========== OPERATIONAL PROPAGATION RISK ==========")
print(risk_summary.round(4))

print("\n========== MODEL SUMMARY ==========")
print(f"Observations scored: {len(operational):,}")
print(f"Mean risk score: {operational['risk_score'].mean():.4f}")
print(f"Minimum risk score: {operational['risk_score'].min():.4f}")
print(f"Maximum risk score: {operational['risk_score'].max():.4f}")

print("\nSTATUS: OPERATIONAL RISK SCORER READY")

========== OPERATIONAL PROPAGATION RISK ==========
           observations  actual_propagation_rate  mean_risk_score
risk_band                                                        
LOW                5360                   0.1965           0.4615
MODERATE          16457                   0.3535           0.4784
ELEVATED          36121                   0.5258           0.5102
HIGH               2131                   0.8282           0.5708
CRITICAL             57                   0.9825           0.6028

========== MODEL SUMMARY ==========
Observations scored: 60,126
Mean risk score: 0.4994
Minimum risk score: 0.4013
Maximum risk score: 0.6097

STATUS: OPERATIONAL RISK SCORER READY


In [53]:
# ============================================================
# CELL 19 — OPERATIONAL WARNING PERFORMANCE
# ============================================================

from sklearn.metrics import confusion_matrix

thresholds = [0.48, 0.50, 0.52, 0.54, 0.56, 0.58, 0.60]

results = []

actual = operational["actual_propagation"].values
scores = operational["risk_score"].values

for threshold in thresholds:

    warnings = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual,
        warnings,
        labels=[0, 1]
    ).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    warning_rate = warnings.mean()

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
    )

    results.append({
        "threshold": threshold,
        "warnings": int(warnings.sum()),
        "warning_rate": warning_rate,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

warning_results = pd.DataFrame(results)

print("========== OPERATIONAL WARNING PERFORMANCE ==========")
print(warning_results.round(4).to_string(index=False))

# ------------------------------------------------------------
# Best F1 threshold
# ------------------------------------------------------------

best = warning_results.loc[
    warning_results["f1"].idxmax()
]

print("\n========== BEST BALANCED THRESHOLD ==========")
print(f"Threshold:     {best['threshold']:.2f}")
print(f"Warnings:      {int(best['warnings']):,}")
print(f"Warning rate:  {best['warning_rate']:.2%}")
print(f"Precision:     {best['precision']:.4f}")
print(f"Recall:        {best['recall']:.4f}")
print(f"F1:            {best['f1']:.4f}")

# ------------------------------------------------------------
# High-confidence operational warning
# ------------------------------------------------------------

high_conf = warning_results[
    warning_results["precision"] >= 0.80
]

print("\n========== HIGH-CONFIDENCE WARNING OPTIONS ==========")

if len(high_conf) > 0:
    print(
        high_conf[
            [
                "threshold",
                "warnings",
                "warning_rate",
                "precision",
                "recall"
            ]
        ].round(4).to_string(index=False)
    )
else:
    print("No threshold achieved >=80% precision.")

print("\nSTATUS: WARNING POLICY EVALUATED")

========== OPERATIONAL WARNING PERFORMANCE ==========
 threshold  warnings  warning_rate  precision  recall     f1
      0.48     42523        0.7072     0.5281  0.8113 0.6398
      0.50     38309        0.6371     0.5433  0.7518 0.6308
      0.52      6108        0.1016     0.7646  0.1687 0.2764
      0.54      2937        0.0488     0.8155  0.0865 0.1564
      0.56      1634        0.0272     0.8525  0.0503 0.0950
      0.58       624        0.0104     0.9311  0.0210 0.0411
      0.60        57        0.0009     0.9825  0.0020 0.0040

========== BEST BALANCED THRESHOLD ==========
Threshold:     0.48
Warnings:      42,523
Warning rate:  70.72%
Precision:     0.5281
Recall:        0.8113
F1:            0.6398

========== HIGH-CONFIDENCE WARNING OPTIONS ==========
 threshold  warnings  warning_rate  precision  recall
      0.54      2937        0.0488     0.8155  0.0865
      0.56      1634        0.0272     0.8525  0.0503
      0.58       624        0.0104     0.9311  0.0210
      0.60

In [56]:
# ============================================================
# CELL 55 — BUILD OPERATIONAL DATA FOR TEMPORAL TEST SET
# ============================================================

# Use the SAME clean temporal test indices that produced
# the 60,126 observations used by the frozen model.

# ------------------------------------------------------------
# 1. Identify the clean temporal test indices
# ------------------------------------------------------------

print("Temporal test rows:", len(test_indices))

# The frozen model was evaluated on X_test_clean.
# Its index is the authoritative index for operational scoring.

operational_indices = X_test_clean.index

print("Clean temporal test rows:", len(operational_indices))

# ------------------------------------------------------------
# 2. Build operational rows from multi_prop
# ------------------------------------------------------------

operational_base = multi_prop.loc[
    operational_indices,
    [
        "service_date",
        "station",
        "source_train",
        "target_train",
        "source_arr_delay",
        "target_arr_delay",
        "gap_minutes",
        "source_delay_class"
    ]
].copy()

# ------------------------------------------------------------
# 3. Align exactly with model features
# ------------------------------------------------------------

X_test_operational = X.loc[operational_indices].copy()

# ------------------------------------------------------------
# 4. Safety checks
# ------------------------------------------------------------

print("\n========== OPERATIONAL DATA ALIGNMENT ==========")
print("Operational rows:", len(operational_base))
print("Model feature rows:", len(X_test_operational))

assert len(operational_base) == len(X_test_operational), \
    "Operational data and model features are still misaligned."

assert operational_base.index.equals(X_test_operational.index), \
    "Operational data and model features have different indices."

print("STATUS: OPERATIONAL DATA ALIGNED")

Temporal test rows: 61432
Clean temporal test rows: 60126

========== OPERATIONAL DATA ALIGNMENT ==========
Operational rows: 60126
Model feature rows: 60126
STATUS: OPERATIONAL DATA ALIGNED


In [18]:
# ============================================================
# CELL 56 — OPERATIONAL RISK TABLE
# ============================================================

# ------------------------------------------------------------
# 1. Get the exact features used by the frozen model
# ------------------------------------------------------------

final_model = clean_temporal_rf

model_features = final_model.feature_names_in_

print("Model expects:", len(model_features), "features")
print("Operational matrix currently has:", len(X_test_operational.columns))

# ------------------------------------------------------------
# 2. Build EXACT model input
# ------------------------------------------------------------

X_test_final = X_test_operational.reindex(
    columns=model_features,
    fill_value=0
)

print("Final prediction matrix:", X_test_final.shape)

# Safety check
assert list(X_test_final.columns) == list(model_features), \
    "Feature columns do not match the frozen model."

# ------------------------------------------------------------
# 3. Generate propagation probabilities
# ------------------------------------------------------------

operational_probs = final_model.predict_proba(
    X_test_final
)[:, 1]

# ------------------------------------------------------------
# 4. Build operational risk table
# ------------------------------------------------------------

operational_risk = operational_base.copy()

operational_risk["risk_score"] = operational_probs

# ------------------------------------------------------------
# 5. Assign risk bands
# ------------------------------------------------------------

def assign_risk_band(score):

    if score < 0.47:
        return "LOW"

    elif score < 0.50:
        return "MODERATE"

    elif score < 0.55:
        return "ELEVATED"

    elif score < 0.60:
        return "HIGH"

    else:
        return "CRITICAL"


operational_risk["risk_band"] = (
    operational_risk["risk_score"]
    .apply(assign_risk_band)
)

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("\n========== OPERATIONAL RISK TABLE ==========")

print(
    "Rows scored:",
    len(operational_risk)
)

print(
    "Mean risk score:",
    round(
        operational_risk["risk_score"].mean(),
        4
    )
)

print(
    "Minimum risk score:",
    round(
        operational_risk["risk_score"].min(),
        4
    )
)

print(
    "Maximum risk score:",
    round(
        operational_risk["risk_score"].max(),
        4
    )
)

print("\n========== RISK BAND DISTRIBUTION ==========")

risk_summary = (
    operational_risk
    .groupby("risk_band")
    .agg(
        observations=("risk_score", "size"),
        mean_risk_score=("risk_score", "mean")
    )
    .reindex(
        [
            "LOW",
            "MODERATE",
            "ELEVATED",
            "HIGH",
            "CRITICAL"
        ]
    )
)

print(risk_summary)

print("\nSTATUS: OPERATIONAL RISK TABLE READY")

Model expects: 2111 features


NameError: name 'X_test_operational' is not defined

In [59]:
# ============================================================
# CELL 57 — OPERATIONAL WARNING ENGINE
# ============================================================

WARNING_THRESHOLD = 0.48

operational_warnings = operational_risk.copy()

# ------------------------------------------------------------
# 1. Generate warning flag
# ------------------------------------------------------------

operational_warnings["warning"] = (
    operational_warnings["risk_score"] >= WARNING_THRESHOLD
).astype(int)

# ------------------------------------------------------------
# 2. Warning priority
# ------------------------------------------------------------

def assign_warning_priority(row):

    score = row["risk_score"]

    if score >= 0.60:
        return "CRITICAL"

    elif score >= 0.56:
        return "VERY HIGH"

    elif score >= 0.52:
        return "HIGH"

    elif score >= 0.48:
        return "MODERATE"

    else:
        return "NO WARNING"


operational_warnings["warning_priority"] = (
    operational_warnings
    .apply(assign_warning_priority, axis=1)
)

# ------------------------------------------------------------
# 3. Sort highest-risk interactions first
# ------------------------------------------------------------

operational_warnings = operational_warnings.sort_values(
    "risk_score",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

warning_count = operational_warnings["warning"].sum()

print("========== OPERATIONAL WARNING ENGINE ==========")

print(
    "Threshold:",
    WARNING_THRESHOLD
)

print(
    "Total interactions:",
    len(operational_warnings)
)

print(
    "Warnings generated:",
    warning_count
)

print(
    "Warning rate:",
    round(
        warning_count / len(operational_warnings),
        4
    )
)

print("\n========== WARNING PRIORITIES ==========")

print(
    operational_warnings["warning_priority"]
    .value_counts()
    .reindex(
        [
            "CRITICAL",
            "VERY HIGH",
            "HIGH",
            "MODERATE",
            "NO WARNING"
        ],
        fill_value=0
    )
)

# ------------------------------------------------------------
# 5. Show highest-risk interactions
# ------------------------------------------------------------

print("\n========== TOP 25 OPERATIONAL WARNINGS ==========")

print(
    operational_warnings[
        [
            "service_date",
            "station",
            "source_train",
            "target_train",
            "source_arr_delay",
            "target_arr_delay",
            "gap_minutes",
            "risk_score",
            "risk_band",
            "warning_priority"
        ]
    ].head(25).to_string(index=False)
)

print("\nSTATUS: OPERATIONAL WARNING ENGINE READY")

========== OPERATIONAL WARNING ENGINE ==========
Threshold: 0.48
Total interactions: 60126
Warnings generated: 42523
Warning rate: 0.7072

========== WARNING PRIORITIES ==========
warning_priority
CRITICAL         57
VERY HIGH      1577
HIGH           4474
MODERATE      36415
NO WARNING    17603
Name: count, dtype: int64

========== TOP 25 OPERATIONAL WARNINGS ==========
service_date station source_train target_train  source_arr_delay  target_arr_delay  gap_minutes  risk_score risk_band warning_priority
  2024-09-17    CSMT        22159        20706              24.0               2.0          3.0    0.609749  CRITICAL         CRITICAL
  2024-09-29    CSMT        12072        12261              28.0               4.0          1.0    0.608273  CRITICAL         CRITICAL
  2024-09-11    CSMT        22105        12859              19.0               3.0          4.0    0.606236  CRITICAL         CRITICAL
  2024-09-21    NDLS        12034         2394               3.0             161.0    

In [60]:
# ============================================================
# CELL 58 — OPERATOR ALERT QUEUE
# ============================================================

# Keep only interactions that actually trigger a warning
alert_queue = operational_warnings[
    operational_warnings["warning"] == 1
].copy()

# ------------------------------------------------------------
# 1. Calculate operational severity score
# ------------------------------------------------------------

alert_queue["severity_score"] = (
    alert_queue["risk_score"] * 100
)

# ------------------------------------------------------------
# 2. Create human-readable alert
# ------------------------------------------------------------

def create_alert(row):

    return (
        f"{row['station']}: Train {row['source_train']} "
        f"may propagate delay to Train {row['target_train']} "
        f"(risk {row['risk_score']:.1%})"
    )


alert_queue["alert_message"] = (
    alert_queue.apply(create_alert, axis=1)
)

# ------------------------------------------------------------
# 3. Rank alerts
# ------------------------------------------------------------

alert_queue = alert_queue.sort_values(
    ["risk_score", "source_arr_delay"],
    ascending=[False, False]
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. Operator-facing columns
# ------------------------------------------------------------

operator_alerts = alert_queue[
    [
        "service_date",
        "station",
        "source_train",
        "target_train",
        "source_arr_delay",
        "target_arr_delay",
        "gap_minutes",
        "risk_score",
        "risk_band",
        "warning_priority",
        "alert_message"
    ]
].copy()

# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("========== OPERATOR ALERT QUEUE ==========")

print("Total alerts:", len(operator_alerts))

print("\nPriority distribution:")

print(
    operator_alerts["warning_priority"]
    .value_counts()
    .reindex(
        [
            "CRITICAL",
            "VERY HIGH",
            "HIGH",
            "MODERATE"
        ],
        fill_value=0
    )
)

print("\n========== TOP 20 OPERATOR ALERTS ==========")

print(
    operator_alerts.head(20).to_string(index=False)
)

print("\nSTATUS: OPERATOR ALERT QUEUE READY")

========== OPERATOR ALERT QUEUE ==========
Total alerts: 42523

Priority distribution:
warning_priority
CRITICAL        57
VERY HIGH     1577
HIGH          4474
MODERATE     36415
Name: count, dtype: int64

========== TOP 20 OPERATOR ALERTS ==========
service_date station source_train target_train  source_arr_delay  target_arr_delay  gap_minutes  risk_score risk_band warning_priority                                                     alert_message
  2024-09-17    CSMT        22159        20706              24.0               2.0          3.0    0.609749  CRITICAL         CRITICAL CSMT: Train 22159 may propagate delay to Train 20706 (risk 61.0%)
  2024-09-29    CSMT        12072        12261              28.0               4.0          1.0    0.608273  CRITICAL         CRITICAL CSMT: Train 12072 may propagate delay to Train 12261 (risk 60.8%)
  2024-09-11    CSMT        22105        12859              19.0               3.0          4.0    0.606236  CRITICAL         CRITICAL CSMT: Trai

In [61]:
# ============================================================
# CELL 59 — NETWORK OPERATIONAL SUMMARY
# ============================================================

# ------------------------------------------------------------
# 1. Station-level operational risk
# ------------------------------------------------------------

station_summary = (
    operational_warnings
    .groupby("station")
    .agg(
        interactions=("risk_score", "size"),
        mean_risk=("risk_score", "mean"),
        high_risk_interactions=(
            "warning",
            "sum"
        ),
        mean_source_delay=(
            "source_arr_delay",
            "mean"
        ),
        mean_gap=(
            "gap_minutes",
            "mean"
        )
    )
)

# Only retain stations with reasonable sample size
station_summary = station_summary[
    station_summary["interactions"] >= 30
].copy()

# Warning rate
station_summary["warning_rate"] = (
    station_summary["high_risk_interactions"]
    / station_summary["interactions"]
)

# ------------------------------------------------------------
# 2. Network bottleneck score
# ------------------------------------------------------------

station_summary["bottleneck_score"] = (
    0.50 * station_summary["mean_risk"]
    + 0.30 * station_summary["warning_rate"]
    + 0.20 * (
        station_summary["mean_source_delay"]
        / station_summary["mean_source_delay"].max()
    )
)

# ------------------------------------------------------------
# 3. Rank stations
# ------------------------------------------------------------

station_summary = station_summary.sort_values(
    "bottleneck_score",
    ascending=False
)

# ------------------------------------------------------------
# 4. Display
# ------------------------------------------------------------

print("========== NETWORK OPERATIONAL SUMMARY ==========")

print(
    "Stations evaluated:",
    len(station_summary)
)

print("\n========== TOP 20 NETWORK BOTTLENECKS ==========")

print(
    station_summary[
        [
            "interactions",
            "mean_risk",
            "warning_rate",
            "high_risk_interactions",
            "mean_source_delay",
            "mean_gap",
            "bottleneck_score"
        ]
    ].head(20).round(4)
)

print("\nSTATUS: NETWORK OPERATIONAL SUMMARY READY")

========== NETWORK OPERATIONAL SUMMARY ==========
Stations evaluated: 447

========== TOP 20 NETWORK BOTTLENECKS ==========
         interactions  mean_risk  warning_rate  high_risk_interactions  mean_source_delay  mean_gap  bottleneck_score
station                                                                                                              
CKP                30     0.5319        1.0000                      30           178.8000   16.2333            0.7659
CHI                40     0.5175        0.9750                      39           104.9000   12.5500            0.6686
ASH                58     0.5100        0.9138                      53           120.3966   15.8448            0.6638
RN                 81     0.5423        0.9753                      79            85.8395   13.3580            0.6597
BINA              232     0.5578        0.9655                     224            62.3793   15.2974            0.6383
DBA                44     0.5076        0.9318    

In [62]:
# ============================================================
# CELL 60 — EXPORT FINAL OPERATIONAL OUTPUTS
# ============================================================

import os

# ------------------------------------------------------------
# 1. Create output directory
# ------------------------------------------------------------

os.makedirs("outputs", exist_ok=True)

# ------------------------------------------------------------
# 2. Export operational risk table
# ------------------------------------------------------------

operational_risk.to_csv(
    "outputs/operational_risk.csv",
    index=False
)

# ------------------------------------------------------------
# 3. Export operator alert queue
# ------------------------------------------------------------

operator_alerts.to_csv(
    "outputs/operator_alert_queue.csv",
    index=False
)

# ------------------------------------------------------------
# 4. Export network bottlenecks
# ------------------------------------------------------------

station_summary.reset_index().to_csv(
    "outputs/network_bottlenecks.csv",
    index=False
)

# ------------------------------------------------------------
# 5. Export top warnings separately
# ------------------------------------------------------------

operational_warnings.sort_values(
    "risk_score",
    ascending=False
).head(100).to_csv(
    "outputs/top_100_warnings.csv",
    index=False
)

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("========== OPERATIONAL OUTPUTS EXPORTED ==========")

print("Risk table:")
print("  outputs/operational_risk.csv")

print("Operator alerts:")
print("  outputs/operator_alert_queue.csv")

print("Network bottlenecks:")
print("  outputs/network_bottlenecks.csv")

print("Top warnings:")
print("  outputs/top_100_warnings.csv")

print("\nSTATUS: OPERATIONAL OUTPUTS SAVED")

========== OPERATIONAL OUTPUTS EXPORTED ==========
Risk table:
  outputs/operational_risk.csv
Operator alerts:
  outputs/operator_alert_queue.csv
Network bottlenecks:
  outputs/network_bottlenecks.csv
Top warnings:
  outputs/top_100_warnings.csv

STATUS: OPERATIONAL OUTPUTS SAVED


In [63]:
# ============================================================
# CELL 61 — FINAL OPERATIONAL SUMMARY
# ============================================================

print("========== FINAL RAILWAY INTELLIGENCE SUMMARY ==========")

print("\nMODEL")
print("Model: Clean Temporal Random Forest")
print("Features:", len(final_model.feature_names_in_))
print("ROC-AUC: 0.6503")

print("\nRISK SCORING")
print("Observations scored:", len(operational_risk))
print(
    "Mean risk score:",
    round(operational_risk["risk_score"].mean(), 4)
)
print(
    "Maximum risk score:",
    round(operational_risk["risk_score"].max(), 4)
)

print("\nRISK BANDS")
print(
    operational_risk["risk_band"]
    .value_counts()
    .sort_index()
)

print("\nOPERATOR ALERTS")
print("Total alerts:", len(operator_alerts))

print(
    operator_alerts["warning_priority"]
    .value_counts()
)

print("\nNETWORK")
print(
    "Stations evaluated:",
    len(station_summary)
)

print("\nTOP 10 BOTTLENECKS")

print(
    station_summary[
        [
            "interactions",
            "mean_risk",
            "warning_rate",
            "mean_source_delay",
            "mean_gap",
            "bottleneck_score"
        ]
    ]
    .head(10)
    .round(4)
)

print("\n========== FINAL SYSTEM STATUS ==========")
print("Propagation model: READY")
print("Risk scorer: READY")
print("Warning engine: READY")
print("Operator alert queue: READY")
print("Network bottleneck analysis: READY")
print("Operational exports: READY")

========== FINAL RAILWAY INTELLIGENCE SUMMARY ==========

MODEL
Model: Clean Temporal Random Forest
Features: 2111
ROC-AUC: 0.6503

RISK SCORING
Observations scored: 60126
Mean risk score: 0.4994
Maximum risk score: 0.6097

RISK BANDS
risk_band
CRITICAL       57
ELEVATED    36121
HIGH         2131
LOW          5360
MODERATE    16457
Name: count, dtype: int64

OPERATOR ALERTS
Total alerts: 42523
warning_priority
MODERATE     36415
HIGH          4474
VERY HIGH     1577
CRITICAL        57
Name: count, dtype: int64

NETWORK
Stations evaluated: 447

TOP 10 BOTTLENECKS
         interactions  mean_risk  warning_rate  mean_source_delay  mean_gap  bottleneck_score
station                                                                                      
CKP                30     0.5319        1.0000           178.8000   16.2333            0.7659
CHI                40     0.5175        0.9750           104.9000   12.5500            0.6686
ASH                58     0.5100        0.9138        

In [65]:
# ============================================================
# CELL 62 — FINAL SYSTEM VALIDATION
# ============================================================

print("========== FINAL SYSTEM VALIDATION ==========")

# ------------------------------------------------------------
# 1. Model validation
# ------------------------------------------------------------

assert final_model is clean_temporal_rf
assert len(final_model.feature_names_in_) == 2111

print("✓ Final model loaded")
print("✓ Model features:", len(final_model.feature_names_in_))

# ------------------------------------------------------------
# 2. Operational data validation
# ------------------------------------------------------------

assert len(operational_risk) == len(X_test_operational)

print("✓ Operational rows aligned")
print("✓ Rows scored:", len(operational_risk))

# ------------------------------------------------------------
# 3. Risk-score validation
# ------------------------------------------------------------

assert operational_risk["risk_score"].between(0, 1).all()

print("✓ Risk scores valid [0,1]")

# ------------------------------------------------------------
# 4. Risk-band validation
# ------------------------------------------------------------

expected_bands = {
    "LOW",
    "MODERATE",
    "ELEVATED",
    "HIGH",
    "CRITICAL"
}

actual_bands = set(
    operational_risk["risk_band"].dropna().unique()
)

assert actual_bands.issubset(expected_bands)

print("✓ Risk bands valid")

# ------------------------------------------------------------
# 5. Warning-engine validation
# ------------------------------------------------------------

assert len(operator_alerts) == 42523

assert (
    operator_alerts["risk_score"] >= 0.48
).all()

print("✓ Warning threshold validated")
print("✓ Alerts:", len(operator_alerts))

# ------------------------------------------------------------
# 6. Network analysis validation
# ------------------------------------------------------------

assert len(station_summary) > 0

assert (
    station_summary["bottleneck_score"]
    .notna()
    .all()
)

print("✓ Network bottleneck analysis valid")
print("✓ Stations evaluated:", len(station_summary))

# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print("\n========== VALIDATION RESULT ==========")
print("MODEL:                 PASS")
print("RISK SCORER:           PASS")
print("WARNING ENGINE:        PASS")
print("OPERATOR ALERT QUEUE:  PASS")
print("NETWORK ANALYSIS:      PASS")
print("DATA ALIGNMENT:        PASS")

print("\nSTATUS: FINAL ML PIPELINE VALIDATED")

========== FINAL SYSTEM VALIDATION ==========
✓ Final model loaded
✓ Model features: 2111
✓ Operational rows aligned
✓ Rows scored: 60126
✓ Risk scores valid [0,1]
✓ Risk bands valid
✓ Warning threshold validated
✓ Alerts: 42523
✓ Network bottleneck analysis valid
✓ Stations evaluated: 447

========== VALIDATION RESULT ==========
MODEL:                 PASS
RISK SCORER:           PASS
WARNING ENGINE:        PASS
OPERATOR ALERT QUEUE:  PASS
NETWORK ANALYSIS:      PASS
DATA ALIGNMENT:        PASS

STATUS: FINAL ML PIPELINE VALIDATED


In [66]:
# ============================================================
# CELL 63 — REUSABLE PROPAGATION PREDICTOR
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Load frozen model + feature schema
# ------------------------------------------------------------

propagation_model = final_model
propagation_features = list(propagation_model.feature_names_in_)

print("========== PROPAGATION PREDICTOR ==========")
print("Model:", type(propagation_model).__name__)
print("Expected features:", len(propagation_features))


# ------------------------------------------------------------
# 2. Prediction function
# ------------------------------------------------------------

def predict_propagation_risk(feature_row):
    """
    Predict delay-propagation risk for one train interaction.

    Parameters
    ----------
    feature_row : pandas.DataFrame
        Must contain the same 2111 features used during model training.

    Returns
    -------
    dict
        Risk score, risk band, warning status and priority.
    """

    # Ensure DataFrame
    if isinstance(feature_row, pd.Series):
        feature_row = feature_row.to_frame().T

    # --------------------------------------------------------
    # Validate required features
    # --------------------------------------------------------

    missing_features = [
        f for f in propagation_features
        if f not in feature_row.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing {len(missing_features)} model features."
        )

    # Keep EXACT training feature order
    X_input = feature_row[
        propagation_features
    ].copy()

    # --------------------------------------------------------
    # Generate probability
    # --------------------------------------------------------

    risk_score = float(
        propagation_model.predict_proba(X_input)[0, 1]
    )

    # --------------------------------------------------------
    # Risk band
    # --------------------------------------------------------

    if risk_score < 0.47:
        risk_band = "LOW"

    elif risk_score < 0.50:
        risk_band = "MODERATE"

    elif risk_score < 0.55:
        risk_band = "ELEVATED"

    elif risk_score < 0.60:
        risk_band = "HIGH"

    else:
        risk_band = "CRITICAL"

    # --------------------------------------------------------
    # Warning policy
    # --------------------------------------------------------

    warning = risk_score >= 0.48

    if not warning:
        priority = "NO WARNING"

    elif risk_score >= 0.60:
        priority = "CRITICAL"

    elif risk_score >= 0.55:
        priority = "VERY HIGH"

    elif risk_score >= 0.50:
        priority = "HIGH"

    else:
        priority = "MODERATE"

    # --------------------------------------------------------
    # Return backend-friendly result
    # --------------------------------------------------------

    return {
        "risk_score": round(risk_score, 4),
        "risk_percentage": round(risk_score * 100, 2),
        "risk_band": risk_band,
        "warning": bool(warning),
        "warning_priority": priority
    }


print("STATUS: PROPAGATION PREDICTOR READY")

========== PROPAGATION PREDICTOR ==========
Model: RandomForestClassifier
Expected features: 2111
STATUS: PROPAGATION PREDICTOR READY


In [3]:
# CELL 64A — REBUILD OPERATIONAL TEST DATA

print("========== REBUILDING OPERATIONAL TEST DATA ==========")

# Columns required for the operational prediction layer
operational_columns = [
    "service_date",
    "station",
    "source_train",
    "target_train",
    "source_arr_delay",
    "target_arr_delay",
    "gap_minutes",
    "source_delay_class"
]

# Rebuild from the propagation dataset
operational_base = multi_prop[operational_columns].copy()

print("operational_base shape:", operational_base.shape)
print("Columns:", list(operational_base.columns))

# ------------------------------------------------------------
# Rebuild feature matrix using the same rows
# ------------------------------------------------------------

operational_indices = operational_base.index

X_test_operational = X.loc[operational_indices].copy()

# Align EXACTLY with the frozen production model
model_features = final_model.feature_names_in_

X_test_final = X_test_operational.reindex(
    columns=model_features,
    fill_value=0
)

print("\nX_test_operational shape:", X_test_operational.shape)
print("X_test_final shape:", X_test_final.shape)
print("Expected model features:", len(model_features))
print("Actual features:", X_test_final.shape[1])

assert X_test_final.shape[1] == len(model_features)

print("\nSTATUS: OPERATIONAL TEST DATA RESTORED")

========== REBUILDING OPERATIONAL TEST DATA ==========


NameError: name 'multi_prop' is not defined

In [1]:
# ============================================================
# CELL 64 — PREDICTOR CONSISTENCY TEST
# ============================================================

print("========== PREDICTOR CONSISTENCY TEST ==========")

# ------------------------------------------------------------
# 1. Take a sample from the already validated test set
# ------------------------------------------------------------

sample_size = 100

X_sample = X_test_final.iloc[:sample_size].copy()

# ------------------------------------------------------------
# 2. Predictions from reusable function
# ------------------------------------------------------------

function_scores = []

for i in range(sample_size):

    result = predict_propagation_risk(
        X_sample.iloc[[i]]
    )

    function_scores.append(
        result["risk_score"]
    )

function_scores = np.array(function_scores)

# ------------------------------------------------------------
# 3. Direct model predictions
# ------------------------------------------------------------

direct_scores = propagation_model.predict_proba(
    X_sample
)[:, 1]

# ------------------------------------------------------------
# 4. Compare
# ------------------------------------------------------------

difference = np.abs(
    function_scores - direct_scores
)

print("Samples tested:", sample_size)
print(
    "Maximum difference:",
    difference.max()
)

print(
    "Mean difference:",
    difference.mean()
)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

assert np.allclose(
    function_scores,
    direct_scores,
    atol=1e-10
)

print("\n✓ Reusable predictor matches frozen model")
print("✓ Feature ordering preserved")
print("✓ Risk scores reproduced exactly")

print("\nSTATUS: PREDICTION FUNCTION VALIDATED")

========== PREDICTOR CONSISTENCY TEST ==========


NameError: name 'X_test_final' is not defined

In [7]:
# CHECK WHICH CORE VARIABLES STILL EXIST

variables = [
    "journey",
    "interaction",
    "pairs",
    "propagation",
    "multi_prop",
    "features",
    "X",
    "final_model",
    "propagation_model"
]

for var in variables:
    print(f"{var:20}:", "✓ EXISTS" if var in globals() else "✗ MISSING")

journey             : ✗ MISSING
interaction         : ✗ MISSING
pairs               : ✗ MISSING
propagation         : ✗ MISSING
multi_prop          : ✗ MISSING
features            : ✗ MISSING
X                   : ✗ MISSING
final_model         : ✗ MISSING
propagation_model   : ✗ MISSING


In [19]:
# ============================================================
# RESTORE OPERATIONAL TEST MATRIX
# ============================================================

print("========== RESTORING OPERATIONAL TEST MATRIX ==========")

# The clean temporal model's test set contains 60,126 rows.
# Use the exact indices from the clean temporal test matrix.

operational_indices = X_test_clean.index

print("Operational test indices:", len(operational_indices))

# Rebuild the operational feature matrix
X_test_operational = X.loc[operational_indices].copy()

print(
    "Operational matrix shape:",
    X_test_operational.shape
)

# ------------------------------------------------------------
# Align EXACTLY with the frozen production model
# ------------------------------------------------------------

model_features = final_model.feature_names_in_

print(
    "Model expects:",
    len(model_features),
    "features"
)

X_test_final = X_test_operational.reindex(
    columns=model_features,
    fill_value=0
)

print(
    "Final model input shape:",
    X_test_final.shape
)

assert X_test_final.shape[1] == len(model_features)

print("\n✓ Operational matrix restored")
print("✓ Feature count aligned")
print("STATUS: X_TEST_FINAL READY")

========== RESTORING OPERATIONAL TEST MATRIX ==========


NameError: name 'X_test_clean' is not defined

In [20]:
# ============================================================
# CHECK TEMPORAL MODEL VARIABLES
# ============================================================

names = [
    "X_train_temporal",
    "X_test_temporal",
    "y_train_temporal",
    "y_test_temporal",
    "X_train_clean_temporal",
    "X_test_clean_temporal",
    "y_train_clean_temporal",
    "y_test_clean_temporal",
    "X_train_clean",
    "X_test_clean",
    "X_clean",
    "final_model"
]

for name in names:
    print(
        f"{name:25}",
        "✓ EXISTS" if name in globals() else "✗ missing"
    )

X_train_temporal          ✗ missing
X_test_temporal           ✗ missing
y_train_temporal          ✗ missing
y_test_temporal           ✗ missing
X_train_clean_temporal    ✗ missing
X_test_clean_temporal     ✗ missing
y_train_clean_temporal    ✗ missing
y_test_clean_temporal     ✗ missing
X_train_clean             ✗ missing
X_test_clean              ✗ missing
X_clean                   ✗ missing
final_model               ✓ EXISTS


In [21]:
# ============================================================
# RESTORE TEMPORAL CLEAN TEST SET
# ============================================================

print("========== RESTORING TEMPORAL TEST SET ==========")

# The temporal split used:
# Sep 1–24  -> training
# Sep 25–30  -> testing

temporal_mask = pd.to_datetime(
    clean_model_features["service_date"]
).dt.day <= 24

train_mask = temporal_mask
test_mask = ~temporal_mask

# Clean model predictors
X_clean_temporal = clean_model_features[
    [
        "source_arr_delay",
        "source_delay_class",
        "target_arr_delay",
        "gap_minutes",
        "station"
    ]
].copy()

y_clean_temporal = clean_model_features[
    "target_propagation_5"
].copy()

# Encode categorical variables
X_clean_temporal = pd.get_dummies(
    X_clean_temporal,
    columns=[
        "source_delay_class",
        "station"
    ],
    drop_first=True
)

# Reproduce the exact temporal rows
X_train_clean_temporal = X_clean_temporal.loc[train_mask]
X_test_clean_temporal = X_clean_temporal.loc[test_mask]

y_train_clean_temporal = y_clean_temporal.loc[train_mask]
y_test_clean_temporal = y_clean_temporal.loc[test_mask]

print("Training rows:", len(X_train_clean_temporal))
print("Testing rows:", len(X_test_clean_temporal))

print(
    "Training propagation rate:",
    round(y_train_clean_temporal.mean(), 4)
)

print(
    "Testing propagation rate:",
    round(y_test_clean_temporal.mean(), 4)
)

print(
    "Features:",
    X_test_clean_temporal.shape[1]
)

========== RESTORING TEMPORAL TEST SET ==========


NameError: name 'clean_model_features' is not defined

In [22]:
# ============================================================
# RESTORE CLEAN MODEL FEATURES
# ============================================================

print("========== RESTORING CLEAN MODEL FEATURES ==========")

clean_model_features = features.copy()

print("Rows:", len(clean_model_features))
print("Columns:", len(clean_model_features.columns))

print("\nRequired columns:")

required_columns = [
    "service_date",
    "source_arr_delay",
    "source_delay_class",
    "target_arr_delay",
    "gap_minutes",
    "station",
    "target_propagation_5"
]

for col in required_columns:
    print(
        f"{col:25}",
        "✓" if col in clean_model_features.columns else "✗"
    )

========== RESTORING CLEAN MODEL FEATURES ==========
Rows: 300628
Columns: 19

Required columns:
service_date              ✓
source_arr_delay          ✓
source_delay_class        ✓
target_arr_delay          ✓
gap_minutes               ✓
station                   ✓
target_propagation_5      ✓


In [23]:
# ============================================================
# RESTORE CLEAN TEMPORAL TEST SET
# ============================================================

print("========== RESTORING CLEAN TEMPORAL TEST SET ==========")

# Sep 1–24 = training
# Sep 25–30 = testing

dates = pd.to_datetime(
    clean_model_features["service_date"]
)

train_mask = dates.dt.day <= 24
test_mask = dates.dt.day >= 25

# ------------------------------------------------------------
# Clean predictors
# ------------------------------------------------------------

X_clean_temporal = clean_model_features[
    [
        "source_arr_delay",
        "source_delay_class",
        "target_arr_delay",
        "gap_minutes",
        "station"
    ]
].copy()

# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

y_clean_temporal = clean_model_features[
    "target_propagation_5"
].copy()

# ------------------------------------------------------------
# Encode categorical variables
# ------------------------------------------------------------

X_clean_temporal = pd.get_dummies(
    X_clean_temporal,
    columns=[
        "source_delay_class",
        "station"
    ],
    drop_first=True
)

# ------------------------------------------------------------
# Temporal split
# ------------------------------------------------------------

X_train_clean_temporal = X_clean_temporal.loc[train_mask]
X_test_clean_temporal = X_clean_temporal.loc[test_mask]

y_train_clean_temporal = y_clean_temporal.loc[train_mask]
y_test_clean_temporal = y_clean_temporal.loc[test_mask]

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("Training rows:", len(X_train_clean_temporal))
print("Testing rows:", len(X_test_clean_temporal))

print(
    "Training propagation rate:",
    round(y_train_clean_temporal.mean(), 4)
)

print(
    "Testing propagation rate:",
    round(y_test_clean_temporal.mean(), 4)
)

print(
    "Encoded features:",
    X_clean_temporal.shape[1]
)

========== RESTORING CLEAN TEMPORAL TEST SET ==========
Training rows: 239196
Testing rows: 61432
Training propagation rate: 0.4586
Testing propagation rate: 0.4675
Encoded features: 2111


In [25]:
# ============================================================
# RESTORE OPERATIONAL TEST MATRIX
# ============================================================

print("========== RESTORING OPERATIONAL TEST MATRIX ==========")

# The operational layer uses the rows that survived
# the interaction/model alignment.

operational_indices = (
    X_test_clean_temporal.index
    .intersection(multi_prop.index)
)

print(
    "Operational indices:",
    len(operational_indices)
)

# Build the operational feature matrix
X_test_operational = X.loc[
    operational_indices
].copy()

print(
    "Operational matrix:",
    X_test_operational.shape
)

# ------------------------------------------------------------
# Align EXACTLY to frozen model
# ------------------------------------------------------------

model_features = final_model.feature_names_in_

X_test_final = X_test_operational.reindex(
    columns=model_features,
    fill_value=0
)

print(
    "Model expects:",
    len(model_features),
    "features"
)

print(
    "Final model input:",
    X_test_final.shape
)

assert X_test_final.shape[1] == len(model_features)

print("\n✓ Operational matrix restored")
print("✓ Feature ordering aligned")
print("STATUS: X_TEST_FINAL READY")

========== RESTORING OPERATIONAL TEST MATRIX ==========
Operational indices: 61432
Operational matrix: (61432, 2113)
Model expects: 2111 features
Final model input: (61432, 2111)

✓ Operational matrix restored
✓ Feature ordering aligned
STATUS: X_TEST_FINAL READY


In [27]:
# ============================================================
# RESTORE CELL 63 — REUSABLE PROPAGATION PREDICTOR
# ============================================================

print("========== RESTORING PROPAGATION PREDICTOR ==========")

# Frozen production model
propagation_model = final_model

# Exact feature order expected by the model
propagation_features = list(
    propagation_model.feature_names_in_
)


def predict_propagation_risk(feature_row):

    # Accept either Series or DataFrame
    if isinstance(feature_row, pd.Series):
        feature_row = feature_row.to_frame().T

    # Check for missing model features
    missing_features = [
        col
        for col in propagation_features
        if col not in feature_row.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing model features: {missing_features[:10]}"
        )

    # Force exact feature ordering
    model_input = feature_row[
        propagation_features
    ].copy()

    # Model prediction
    risk_score = propagation_model.predict_proba(
        model_input
    )[0, 1]

    # Risk band
    if risk_score < 0.47:
        risk_band = "LOW"

    elif risk_score < 0.50:
        risk_band = "MODERATE"

    elif risk_score < 0.55:
        risk_band = "ELEVATED"

    elif risk_score < 0.60:
        risk_band = "HIGH"

    else:
        risk_band = "CRITICAL"

    # Warning
    warning = risk_score >= 0.48

    # Operator priority
    if risk_score >= 0.60:
        warning_priority = "CRITICAL"

    elif risk_score >= 0.55:
        warning_priority = "VERY HIGH"

    elif risk_score >= 0.50:
        warning_priority = "HIGH"

    elif risk_score >= 0.48:
        warning_priority = "MODERATE"

    else:
        warning_priority = "NO WARNING"

    return {
        "risk_score": risk_score,
        "risk_percentage": risk_score * 100,
        "risk_band": risk_band,
        "warning": warning,
        "warning_priority": warning_priority
    }


print(
    "Model features:",
    len(propagation_features)
)

print(
    "Model:",
    type(propagation_model).__name__
)

print("\n✓ Reusable predictor restored")
print("STATUS: PREDICTOR READY")

========== RESTORING PROPAGATION PREDICTOR ==========
Model features: 2111
Model: RandomForestClassifier

✓ Reusable predictor restored
STATUS: PREDICTOR READY


In [28]:
# ============================================================
# CELL 64 — PREDICTOR CONSISTENCY TEST
# ============================================================

print("========== PREDICTOR CONSISTENCY TEST ==========")

sample_size = 100

X_sample = X_test_final.iloc[:sample_size].copy()

function_scores = []

for i in range(sample_size):

    result = predict_propagation_risk(
        X_sample.iloc[[i]]
    )

    function_scores.append(
        result["risk_score"]
    )

function_scores = np.array(function_scores)

direct_scores = propagation_model.predict_proba(
    X_sample
)[:, 1]

difference = np.abs(
    function_scores - direct_scores
)

print("Samples tested:", sample_size)
print(
    "Maximum difference:",
    difference.max()
)

print(
    "Mean difference:",
    difference.mean()
)

assert np.allclose(
    function_scores,
    direct_scores,
    atol=1e-10
)

print("\n✓ Reusable predictor matches frozen model")
print("✓ Feature ordering preserved")
print("✓ Risk scores reproduced exactly")

print("\nSTATUS: PREDICTION FUNCTION VALIDATED")

========== PREDICTOR CONSISTENCY TEST ==========
Samples tested: 100
Maximum difference: 2.220446049250313e-16
Mean difference: 5.60662627435704e-17

✓ Reusable predictor matches frozen model
✓ Feature ordering preserved
✓ Risk scores reproduced exactly

STATUS: PREDICTION FUNCTION VALIDATED


In [29]:
# ============================================================
# CELL 65 — BACKEND-READY RAW INPUT PREDICTOR
# ============================================================

print("========== RAW INPUT PROPAGATION PREDICTOR ==========")


def predict_propagation_from_raw(
    station,
    source_arr_delay,
    source_delay_class,
    gap_minutes
):
    """
    Backend-friendly propagation risk predictor.

    Inputs:
        station              : railway station code
        source_arr_delay     : current source train arrival delay
        source_delay_class   : LOW / MODERATE / HIGH / SEVERE
        gap_minutes          : temporal gap between trains

    Returns:
        Propagation risk score, band, warning and priority.
    """

    # --------------------------------------------------------
    # 1. Create raw feature row
    # --------------------------------------------------------

    raw = pd.DataFrame([{
        "source_arr_delay": source_arr_delay,
        "source_delay_class": source_delay_class,
        "gap_minutes": gap_minutes,
        "station": station
    }])

    # --------------------------------------------------------
    # 2. Encode categorical variables
    # --------------------------------------------------------

    encoded = pd.get_dummies(
        raw,
        columns=[
            "source_delay_class",
            "station"
        ],
        drop_first=True
    )

    # --------------------------------------------------------
    # 3. Align with exact frozen model features
    # --------------------------------------------------------

    encoded = encoded.reindex(
        columns=propagation_features,
        fill_value=0
    )

    # --------------------------------------------------------
    # 4. Use validated predictor
    # --------------------------------------------------------

    result = predict_propagation_risk(encoded)

    # --------------------------------------------------------
    # 5. Add input information for API/backend use
    # --------------------------------------------------------

    result["station"] = station
    result["source_arr_delay"] = source_arr_delay
    result["source_delay_class"] = source_delay_class
    result["gap_minutes"] = gap_minutes

    return result


print("✓ Raw-input wrapper created")
print("STATUS: BACKEND INTERFACE READY")

========== RAW INPUT PROPAGATION PREDICTOR ==========
✓ Raw-input wrapper created
STATUS: BACKEND INTERFACE READY


In [30]:
# ============================================================
# CELL 66 — BACKEND INTERFACE TEST
# ============================================================

print("========== BACKEND INTERFACE TEST ==========")

test_result = predict_propagation_from_raw(
    station="CSMT",
    source_arr_delay=24,
    source_delay_class="HIGH",
    gap_minutes=3
)

print("\nPrediction result:")

for key, value in test_result.items():
    print(f"{key}: {value}")

print("\nSTATUS: RAW INPUT TEST COMPLETE")

========== BACKEND INTERFACE TEST ==========

Prediction result:
risk_score: 0.4706377052515208
risk_percentage: 47.06377052515208
risk_band: MODERATE
warning_priority: NO WARNING
station: CSMT
source_arr_delay: 24
source_delay_class: HIGH
gap_minutes: 3

STATUS: RAW INPUT TEST COMPLETE


In [31]:
# ============================================================
# CELL 67 — HISTORICAL INTERACTION VALIDATION
# ============================================================

print("========== HISTORICAL INTERACTION VALIDATION ==========")

# Take the first valid historical interaction
row = multi_prop.iloc[0]

# ------------------------------------------------------------
# 1. Predict using backend-style raw inputs
# ------------------------------------------------------------

raw_result = predict_propagation_from_raw(
    station=row["station"],
    source_arr_delay=row["source_arr_delay"],
    source_delay_class=row["source_delay_class"],
    gap_minutes=row["gap_minutes"]
)

# ------------------------------------------------------------
# 2. Build equivalent encoded row
# ------------------------------------------------------------

raw = pd.DataFrame([{
    "source_arr_delay": row["source_arr_delay"],
    "source_delay_class": row["source_delay_class"],
    "gap_minutes": row["gap_minutes"],
    "station": row["station"]
}])

encoded = pd.get_dummies(
    raw,
    columns=[
        "source_delay_class",
        "station"
    ],
    drop_first=True
)

encoded = encoded.reindex(
    columns=propagation_features,
    fill_value=0
)

direct_result = predict_propagation_risk(encoded)

# ------------------------------------------------------------
# 3. Compare
# ------------------------------------------------------------

difference = abs(
    raw_result["risk_score"]
    -
    direct_result["risk_score"]
)

print("\nHistorical interaction:")
print("Station:", row["station"])
print("Source train:", row["source_train"])
print("Target train:", row["target_train"])
print("Source delay:", row["source_arr_delay"])
print("Delay class:", row["source_delay_class"])
print("Gap:", row["gap_minutes"], "minutes")

print("\nWrapper risk:", raw_result["risk_score"])
print("Direct risk:", direct_result["risk_score"])
print("Difference:", difference)

assert difference < 1e-10

print("\n✓ Historical interaction processed")
print("✓ Raw wrapper matches direct predictor")
print("✓ Backend interface validated")

print("\nSTATUS: HISTORICAL VALIDATION PASSED")

========== HISTORICAL INTERACTION VALIDATION ==========

Historical interaction:
Station: AADR
Source train: 14054
Target train: 12057
Source delay: 2.0
Delay class: LOW
Gap: 26.0 minutes

Wrapper risk: 0.4755057872760171
Direct risk: 0.4755057872760171
Difference: 0.0

✓ Historical interaction processed
✓ Raw wrapper matches direct predictor
✓ Backend interface validated

STATUS: HISTORICAL VALIDATION PASSED
